# 0 から作る世界モデル (1) Mesa の住民の目線

住民の目線の画像と「前進 / 左 / 右 / 後退」の行動から、**次に何が見えるか**を予測する小さな世界モデルを 0 から学習します。
形は DINO-WM (2024) と同じです。

```
目線の画像 ──DINOv2 (凍結)──> 特徴 z_t ─┐
                                        ├─ 世界モデル (Transformer) ─> 次の特徴 z_{t+1}  ← 順動力学 = シミュレータ
行動 a_t (前進/左/右/後退) ─────────────┘
(z_t, z_{t+1}) ─ 逆動力学 ─> a_t                                                       ← 映像 → 行動
ゴールの画像の特徴 ─ 想像の中で行動を試す (CEM) ─> 次の一手                            ← 計画
```

データは、persona の学習で使ってきた地図の生成器と、[mesa-clouds-renderer](https://github.com/oggata/mesa-clouds-renderer) の 3D 表示に寄せた
テクスチャ付きレイキャスタで、その場で作ります。建物は renderer と同じ写真 (`textures/v4`) と業種ごとの高さ (2x2 の大きな建物も)、
地面はアスファルト・歩道と路面標示・芝生・土 (`textures/base`, `textures/road/road_atlas.png`)、それに木と昼の空を描きます。
本物の描画器が手元にあるので、**想像した未来と実際の未来を並べて見比べられ**、計画した行動を本物の環境で試せます。

- ランタイム: **A100** (L4 でも動くが、`"sharp"` 以上はメモリと時間がきつい)
- 学習の量と画質は、すぐ下の**設定のセルだけ**で変えられます (`PRESET` を選ぶ。今までと同じ速さは `"fast"` で A100 約 10 分)
- ダウンロードするもの (DINOv2 と VGG16 の重み、街の写真) と、学習したモデルの書き出しは、すべて **Google Drive** の `MyDrive/mesa_world_model/` に置きます。
  最初に Drive への接続の許可を求められます。2 回目からは Drive にあるものを使うので、ダウンロードし直しません
- 学習に使う地図と、評価に使う地図は分けてあります (評価は**見たことのない街**で行う)

In [ ]:
!apt-get -qq install -y fonts-noto-cjk > /dev/null
import os, math, random, time, json, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# 日本語の見出しを出すためのフォント (Colab は apt で入れた Noto、Mac はヒラギノ)
import matplotlib.font_manager as fm
for f in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    fm.fontManager.addfont(f)
_have = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ("Noto Sans CJK JP", "Hiragino Sans") if f in _have] + ["DejaVu Sans"]

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
QUICK = os.environ.get("WM_QUICK") == "1"   # 手元の動作確認用。Colab では False のまま
AMP = dict(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda")
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 保存先 (Google Drive)

ダウンロードするものと作ったものは、すべて Google Drive の `WORK_DIR` に置きます。ランタイムを切っても消えず、2 回目からはダウンロードし直しません。
実行すると Drive への接続の許可を求められるので、画面の案内どおりに許可してください。

| 置き場所 (`WORK_DIR` の下) | 中身 |
| --- | --- |
| `torch_hub/` | DINOv2 (特徴を取り出す) と VGG16 (復元器の「見た目の近さ」) の重みとコード |
| `mesa_assets/` | 街を描くのに使う写真 (mesa-clouds-renderer の `textures/`) |
| `export_<PRESET>/` | 学習したモデルの書き出し (最後のセル)。PRESET ごとに分けるので、別の設定で回しても上書きしない |

学習データ (描いた画像と DINOv2 の特徴) は、sharp で 14 GB ほどあり、毎回その場で作り直すので Drive には置きません。

In [ ]:
# ── 保存先: ダウンロードするもの・作ったものは、すべてここに置く ──
USE_DRIVE = True                                   # False にすると Drive を使わず、Colab の一時的な場所に置く (ランタイムを切ると消える)
WORK_DIR = "/content/drive/MyDrive/mesa_world_model"

try:
    from google.colab import drive as _gdrive
except ImportError:                                # Colab の外 (手元の Mac など)
    _gdrive = None
if USE_DRIVE and _gdrive is not None:
    _gdrive.mount("/content/drive")
else:
    WORK_DIR = os.environ.get("WM_WORK_DIR") or os.path.abspath("mesa_world_model")
os.makedirs(WORK_DIR, exist_ok=True)
torch.hub.set_dir(os.path.join(WORK_DIR, "torch_hub"))   # DINOv2 (torch.hub.load) と VGG16 (torchvision の重み) はここに落ちる
print("保存先:", WORK_DIR)
for name in ("torch_hub", "mesa_assets"):
    d = os.path.join(WORK_DIR, name)
    n = sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0
    print(f"  {name}/: " + (f"{n} ファイルあり (ダウンロードし直さない)" if n else "まだ無い (このあと落とす)"))

## 設定

学習の量と画質はこのセルで決めます。`PRESET` を選ぶだけで、関係する値がまとめて変わります。
個別に変えたいときは `OVERRIDE` に書きます (例: `OVERRIDE = dict(WM_STEPS=40000)`)。

想像がぼやける原因は 2 つあり、それぞれに効く設定があります。

| 原因 | 効く設定 |
| --- | --- |
| 特徴を 8×8 に粗くしてから 64×64 の画像に戻している (復元の上限が低い) | `TOK_GRID = 16`, `THUMB = 128`, `DEC_PERCEPTUAL`, `DEC_STEPS` |
| 世界モデルの予測がずれ、ずれた予測を使い回すうちに積もる | `WM_D`, `WM_DEPTH`, `WM_STEPS`, `ROLLOUT`, `EP_PER_MAP` |

復元の上限は「4. 学習する」の復元器のセルの絵 (実際 / 特徴から復元) で、世界モデルのずれは「1. どのくらい先まで当たるか」のグラフで見られます。

In [ ]:
# ═══════════════ 設定: 変えるのはこのセルだけ ═══════════════
# PRESET を選ぶと、学習の量と画質がまとめて変わる。時間は A100 での大まかな目安
# (実際の残り時間は、学習中のログに「残り約 ○ 分」と出る)。
#   "fast"    … 前と同じ。約 10 分。想像は 8×8 の特徴から 64×64 に戻すので、ぼやける
#   "sharp"   … 特徴を粗くせず 16×16 のまま使い、128×128 に戻す。世界モデルも大きくし、長く学習する。40〜60 分
#   "sharper" … さらに大きなモデルとデータで、もっと長く学習する。1.5〜2.5 時間 (A100 40GB でメモリぎりぎり)
PRESET = "sharp"

_COLS = ("fast", "sharp", "sharper")
_TABLE = dict(
    # ── データ (学習に使うフレーム数 = 街の数 × 歩き回る回数 × 歩数) ──
    N_MAPS_TRAIN   = (64,    64,    64),     # 学習に使う街の数
    N_MAPS_TEST    = (16,    16,    16),     # 評価だけに使う街の数 (学習では一度も見せない)
    EP_PER_MAP     = (6,     8,     10),     # 1 つの街を何回歩き回るか。増やすとデータが増え、GPU のメモリも増える
    EP_LEN         = (100,   100,   100),    # 1 回に歩く歩数
    # ── 細かさ (ぼやけ具合にいちばん効く) ──
    TOK_GRID       = (8,     16,    16),     # DINOv2 の特徴を何×何のトークンで持つか。16 = 粗くしない (8 の 4 倍の情報。学習は 4〜5 倍重い)
    THUMB          = (64,    128,   128),    # 特徴を画像に戻すときの大きさ (TOK_GRID の 4 倍以上の 2 の累乗倍)
    # ── 世界モデル (Transformer) ──
    WM_D           = (256,   384,   512),    # 幅
    WM_DEPTH       = (6,     6,     8),      # 層の数
    WM_HEADS       = (8,     8,     8),      # 注意のヘッド数 (WM_D を割り切れる数)
    WM_STEPS       = (12000, 20000, 30000),  # 学習を回す回数
    WM_BATCH       = (64,    32,    32),     # 1 回に見る例の数 (16×16 はトークンが多いので小さめ)
    WM_LR          = (3e-4,  3e-4,  2e-4),   # 学習率
    ROLLOUT        = (1,     2,     2),      # 自分の予測を次の入力にして何歩先まで当てる練習をするか。
                                             #   2 以上で、長く想像したときのぼやけ・崩れが減る (その分遅くメモリも食う)
    # ── 復元器 (特徴 → 画像。想像を目で見るためだけに使う) ──
    DEC_CH         = (128,   128,   192),    # 幅
    DEC_STEPS      = (3000,  10000, 20000),  # 学習を回す回数
    DEC_BATCH      = (64,    64,    64),
    DEC_PERCEPTUAL = (0.0,   0.2,   0.2),    # 見た目の近さ (VGG16 の特徴) の損失の重み。0 = 画素の差 (L1) だけ。
                                             #   画素の差だけだと「平均的な絵」を出してぼやけるので、輪郭や模様を合わせさせる
    DEC_GAN        = (0.0,   0.1,   0.1),    # 本物らしさの判定役 (PatchGAN) の損失の重み。0 = 使わない。くっきりさせるのにいちばん効く。
                                             #   実際の重みは毎回、画素の差の勾配と釣り合うよう自動で調整する (学習の後半だけ使う)
    DEC_CTX        = (False, True,  True),   # 直前に見た本物の画像も手がかりに渡す (壁の模様や色をそこから写せる)
    DEC_NOISE      = (0.0,   0.3,   0.3),    # 学習中、特徴に混ぜるノイズの大きさ (想像した特徴のずれに慣らす)
    DEC_IMAGINED   = (0.0,   0.25,  0.25),   # 学習の例のうち、世界モデルが想像した特徴を使う割合 (想像を見るときの崩れが減る)
    DEC_RES        = (0,     1,     2),      # 広げる段ごとの残差ブロックの数 (0 = 前の版と同じ形)
    DEC_EMA        = (0.0,   0.999, 0.999),  # 重みの移動平均 (最後の重みのばらつきを抑える)。0 = 使わない
    # ── メモリ ──
    THUMBS_ON_CPU  = (False, True,  True),   # 縮小画像 (復元器の答え) を GPU ではなく CPU のメモリに置く。
                                             #   1 回に使う分だけ送るので、速さはほぼ変わらず、GPU のメモリが数 GB〜十数 GB 空く
    # ── ほかの住民 (NPC) ──
    NPCS           = (5,     5,     5),      # 街を歩くほかの住民の数。0 = いない (前の版と同じ)。5 などにすると、
                                             #   住民の位置を数の並び (テンソル) で世界モデルに渡して一緒に学習する
    # ── 逆動力学・計画 ──
    INV_STEPS      = (3000,  3000,  3000),   # 逆動力学の学習を回す回数
    N_TRIALS       = (20,    10,    10),     # 計画を試す回数 (いちばん時間がかかるセル。世界モデルが大きいほど遅いので減らしてある)
    POP            = (256,   128,   128),    # 計画で 1 回に想像の中で試す行動の並びの数
)
OVERRIDE = dict()    # 1 つだけ変えたいときはここに書く。例: dict(WM_STEPS=40000, EP_PER_MAP=12)

CFG = {k: v[_COLS.index(PRESET)] for k, v in _TABLE.items()} | OVERRIDE
if QUICK:            # 手元の動作確認用: 量だけ小さくする (形と大きさは PRESET のまま)
    CFG |= dict(N_MAPS_TRAIN=4, N_MAPS_TEST=2, EP_PER_MAP=2, EP_LEN=20, WM_STEPS=60, DEC_STEPS=60,
                INV_STEPS=100, N_TRIALS=2, POP=16)
globals().update(CFG)

# ── ここから下は世界の決まりごと。普通は変えない ──
GRID = 30                    # 地図の大きさ (persona の学習と同じ)
IMG = 224                    # 描画の大きさ (DINOv2 の入力)
N_TOK = TOK_GRID ** 2
MOVE = 0.2                   # 1 歩で進む距離 (マス)
ROT = math.radians(12)       # 1 歩で曲がる角度
ACTIONS = ["前進", "左", "右", "後退"]   # 0〜2 は既存方策と同じ (0=forward 1=turn_left 2=turn_right)。3=後退はこの世界モデル用に足したもの
NA = len(ACTIONS)
HIST = 3                     # 世界モデルが見る過去のフレーム数

_frames = N_MAPS_TRAIN * EP_PER_MAP * (EP_LEN + 1) + N_MAPS_TEST * 4 * (EP_LEN + 1)
print(f"PRESET = {PRESET!r}" + (" (QUICK)" if QUICK else ""))
print("  " + ", ".join(f"{k}={v}" for k, v in CFG.items()))
_gb_feat, _gb_thumb = _frames * N_TOK * 384 * 2 / 1e9, _frames * THUMB * THUMB * 3 / 1e9
print(f"  学習と評価のフレーム {_frames:,} 枚。特徴で GPU のメモリを約 {_gb_feat:.1f} GB、縮小画像で "
      + (f"CPU のメモリを約 {_gb_thumb:.1f} GB 使う" if THUMBS_ON_CPU else f"GPU のメモリを約 {_gb_thumb:.1f} GB 使う"))


## 共通の部品 (DINOv2・世界モデル・復元器・逆動力学)

In [ ]:
# 凍結した DINOv2 (vits14)。224×224 の画像 → 16×16 のパッチ特徴 (各 384 次元)。
# 学習はしない。世界モデルはこの特徴の空間で「次に何が見えるか」を予測する。
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEVICE).eval()
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
FEAT_DIM = 384

@torch.no_grad()
def dino_encode(imgs, grid):
    """imgs: (B,3,224,224) float [0,1] → (B, grid*grid, 384)。grid<16 なら平均プーリングで粗くする"""
    with torch.autocast(**AMP):
        p = dino.forward_features((imgs - IMNET_MEAN) / IMNET_STD)["x_norm_patchtokens"]
    B = p.shape[0]
    p = p.float().transpose(1, 2).reshape(B, FEAT_DIM, 16, 16)
    if grid != 16:
        p = F.adaptive_avg_pool2d(p, grid)
    return p.flatten(2).transpose(1, 2)

In [ ]:
class WorldModel(nn.Module):
    """順動力学: 「過去 T フレームの特徴 + 各フレームでの行動」→「それぞれの次のフレームの特徴」。

    1 フレーム = N 個のパッチトークン。行動の埋め込みを全トークンに足し、フレーム単位の因果マスクで
    「フレーム i はフレーム 0..i だけを見る」ようにする (DINO-WM と同じ形)。
    出力は「前のフレームからの差分」。最初は差分 0 = 「何も変わらない」と予測する状態から学習が始まる。

    n_npc > 0 のときは、ほかの住民 (NPC) の数の並び (B,T,n_npc,npc_f) も受け取る。1 人 = 1 トークンにして、
    各フレームの画像のトークンの後ろに並べる (位置 t には「次のフレーム t+1 で住民がどこにいるか」を入れる。
    住民の動きはシミュレーターが決めるので、世界モデルは「そこにいるときの見た目」を描くだけでよい)。
    n_npc = 0 なら前の版とまったく同じ形 (重みの名前も同じ)。
    """
    def __init__(self, feat_dim, n_tok, hist, act_dim=None, n_actions=None, d=256, depth=6, heads=8, n_npc=0, npc_f=6):
        super().__init__()
        self.hist, self.n_tok, self.n_npc = hist, n_tok, n_npc
        self.inp = nn.Linear(feat_dim, d)
        if n_actions is not None:
            self.act = nn.Embedding(n_actions, d)
        else:
            self.act = nn.Sequential(nn.Linear(act_dim, d), nn.GELU(), nn.Linear(d, d))
        self.pos_tok = nn.Parameter(torch.randn(1, 1, n_tok, d) * 0.02)
        self.pos_t = nn.Parameter(torch.randn(1, hist, 1, d) * 0.02)
        if n_npc:
            self.npc = nn.Sequential(nn.Linear(npc_f, d), nn.GELU(), nn.Linear(d, d))
            self.npc_id = nn.Parameter(torch.randn(1, 1, n_npc, d) * 0.02)      # 何番目の住民か (同じ人をフレームをまたいで追う)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, feat_dim)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
        f = torch.arange(hist).repeat_interleave(n_tok + n_npc)
        self.register_buffer("mask", f[None, :] > f[:, None], persistent=False)  # True = 見てはいけない

    def forward(self, z, a, n=None):
        # z: (B,T,N,D) / a: (B,T) 行動番号 か (B,T,A) 連続値 / n: (B,T,K,F) 住民 (無ければ「誰もいない」)。位置 t の出力が t+1 の予測
        B, T, N, D = z.shape
        h = self.inp(z) + self.pos_tok + self.act(a)[:, :, None, :]
        if self.n_npc:
            if n is None:
                n = torch.zeros(B, T, self.n_npc, self.npc[0].in_features, device=z.device)
            h = torch.cat([h, self.npc(n.to(h.dtype)) + self.npc_id + self.act(a)[:, :, None, :]], 2)
        h = h + self.pos_t[:, -T:]
        L = h.shape[2]
        h = self.blocks(h.reshape(B, T * L, -1), mask=self.mask[:T * L, :T * L])
        return z + self.out(self.norm(h)).reshape(B, T, L, -1)[:, :, :N]


@torch.no_grad()
def imagine(model, z_ctx, a_seq, n_seq=None):
    """想像の中で先へ進める (シミュレータとしての使い方)。
    z_ctx: (B,C,N,D) 実際に見た直近 C フレーム / a_seq: (B, C-1+K, ...) 文脈の先頭フレームからの行動
    n_seq: (B, C-1+K, n_npc, F) 各行動の「あと」のフレームでの住民 (a_seq と同じ並び。無ければ住民なし)
    返り値: (B,K,N,D) 想像した K フレーム先までの特徴。予測を次の入力に使い回すので、先ほど誤差が積もる"""
    H = model.hist
    zs = list(z_ctx.float().unbind(1)); C = len(zs)
    K = a_seq.shape[1] - (C - 1)
    for _ in range(K):
        t = len(zs) - 1; lo = max(0, t - H + 1)
        with torch.autocast(**AMP):
            pred = model(torch.stack(zs[lo:], 1), a_seq[:, lo:t + 1],
                         None if n_seq is None else n_seq[:, lo:t + 1])[:, -1]
        zs.append(pred.float())
    return torch.stack(zs[C:], 1)


class ResBlock(nn.Module):
    """残差ブロック: 入力に「少しの直し」を足す。深くしても学習が崩れにくい"""
    def __init__(self, c):
        super().__init__()
        self.b = nn.Sequential(nn.GroupNorm(8, c), nn.GELU(), nn.Conv2d(c, c, 3, 1, 1),
                               nn.GroupNorm(8, c), nn.GELU(), nn.Conv2d(c, c, 3, 1, 1))

    def forward(self, x):
        return x + self.b(x)


class Decoder(nn.Module):
    """特徴 (grid×grid トークン) → 画像 (3, out, out)。想像した未来を目で見るためだけに使う。
    世界モデルの学習や計画には一切使わない (画素ではなく特徴で考えるのが DINO-WM の要点)。
    out は grid の 2 の累乗倍 (8→64, 16→128 など)。2 倍に広げる層を、その回数だけ重ねる。

    res: 広げる段ごとに重ねる残差ブロックの数。ctx: 直前に見た本物の画像 (手がかり) も受け取る。
    res=0, ctx=False のときは前の版とまったく同じ形 (重みの名前も同じ) なので、前に保存した重みも読める。"""
    def __init__(self, feat_dim, grid, ch=128, out=None, res=0, ctx=False):
        super().__init__()
        self.grid = grid
        self.out_px = out = out or 8 * grid
        self.n_up = n_up = round(math.log2(out / grid))
        assert n_up >= 2 and grid * 2 ** n_up == out, f"out ({out}) は grid ({grid}) の 4 倍以上の 2 の累乗倍にする"
        def up(i, o):
            return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.GroupNorm(8, o), nn.GELU(),
                                 nn.Conv2d(o, o, 3, 1, 1), nn.GroupNorm(8, o), nn.GELU())
        chs = [2 * ch] * (n_up - 1) + [ch, ch // 2]       # 広げるたびに細くする (8→64 なら 2ch, 2ch, ch, ch/2)
        self.ctx = ctx
        if not res and not ctx:                            # 前の版と同じ形
            self.net = nn.Sequential(nn.Conv2d(feat_dim, chs[0], 1), *[up(chs[k], chs[k + 1]) for k in range(n_up)],
                                     nn.Conv2d(ch // 2, 3, 3, 1, 1))
            return
        ce = ch // 2 if ctx else 0                          # 手がかりの画像から取り出す特徴の幅
        if ctx:
            # 手がかりの画像を、out → out/2 → … → grid と縮めながら特徴にする (各段の大きさで、復元の途中に足す)
            self.cenc = nn.ModuleList([nn.Sequential(nn.Conv2d(3, ce, 3, 1, 1), nn.GELU())] +
                                      [nn.Sequential(nn.Conv2d(ce, ce, 4, 2, 1), nn.GroupNorm(8, ce), nn.GELU())
                                       for _ in range(n_up)])
        self.stem = nn.Conv2d(feat_dim, chs[0], 1)
        self.ups = nn.ModuleList([up(chs[k] + ce, chs[k + 1]) for k in range(n_up)])
        self.blocks = nn.ModuleList([nn.Sequential(*[ResBlock(chs[k + 1]) for _ in range(res)]) for k in range(n_up)])
        self.head = nn.Sequential(nn.Conv2d(ch // 2 + ce, ch // 2, 3, 1, 1), nn.GELU(), nn.Conv2d(ch // 2, 3, 3, 1, 1))

    def forward(self, z, ctx_img=None):
        """z: (B,N,D) / ctx_img: (B,3,out,out) [0,1] の手がかりの画像 (ctx=True のとき。None なら手がかりなし)"""
        B, N, D = z.shape
        h = z.transpose(1, 2).reshape(B, D, self.grid, self.grid)
        if not hasattr(self, "stem"):
            return torch.sigmoid(self.net(h))
        h = self.stem(h)
        if self.ctx:
            if ctx_img is None:
                ctx_img = torch.zeros(B, 3, self.out_px, self.out_px, device=z.device)
            if ctx_img.shape[-1] != self.out_px:
                ctx_img = F.interpolate(ctx_img.float(), size=self.out_px, mode="bilinear", antialias=True)
            e = [self.cenc[0](ctx_img.to(h.dtype) if h.dtype != torch.float32 else ctx_img.float())]
            for m in self.cenc[1:]:
                e.append(m(e[-1]))
            e = e[::-1]                                    # e[0] = grid の大きさ … e[-1] = out の大きさ
        for k in range(self.n_up):
            if self.ctx:
                h = torch.cat([h, e[k].to(h.dtype)], 1)
            h = self.blocks[k](self.ups[k](h))
        if self.ctx:
            h = torch.cat([h, e[-1].to(h.dtype)], 1)
        return torch.sigmoid(self.head(h))


class InverseDynamics(nn.Module):
    """逆動力学: (今のフレーム, 次のフレーム) の特徴 → その間の行動。「映像 → 行動 (座標)」の部分"""
    def __init__(self, feat_dim, n_tok, out_dim, r=32):
        super().__init__()
        self.red = nn.Linear(feat_dim, r)
        self.mlp = nn.Sequential(nn.Linear(2 * n_tok * r, 512), nn.GELU(),
                                 nn.Linear(512, 512), nn.GELU(), nn.Linear(512, out_dim))

    def forward(self, z0, z1):
        return self.mlp(torch.cat([self.red(z0).flatten(1), self.red(z1).flatten(1)], 1))


def valid_starts(ep_start, ep_len, H):
    """窓 (最大 H+1 フレーム) が 1 つのエピソードに収まる開始位置の一覧"""
    return torch.cat([torch.arange(s, s + n - H) for s, n in zip(ep_start, ep_len) if n > H])


def eta(t0, done, total):
    """経過と残りの時間の表示"""
    el = time.time() - t0
    return f"{el/60:.1f} 分経過 / 残り約 {el / done * (total - done) / 60:.0f} 分"


def train_wm(model, feats, acts, starts, steps, batch=64, lr=3e-4, rollout=1, log_every=500, npc=None):
    """feats: (M,N,D) 全フレームを並べた特徴 / acts: (M,) か (M,A) フレーム i で取った行動
    starts: valid_starts(..., hist + rollout - 1)
    rollout>1 なら、1 歩先を当てたあと、その予測を次の入力にしてさらに先 (rollout 歩先まで) も当てさせる。
    想像では予測を使い回すので、その使い方で練習しておくと、ずれが積もりにくくなる。
    npc: (M,K,F) 各フレームでの住民の数の並び (無ければ None)。位置 t には次のフレーム t+1 の住民を渡す"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.05)
    H = model.hist; losses, far = [], []; t0 = time.time()
    model.train()
    for step in range(steps):
        T = random.randint(1, H)                       # 文脈の長さもばらす (始めのうちは過去が短いので)
        i = starts[torch.randint(0, len(starts), (batch,))]
        idx = (i[:, None] + torch.arange(T + rollout)).to(feats.device)
        z = feats[idx].float()                          # (B,T+R,N,D)
        a = acts[idx[:, :T + rollout - 1]]              # (B,T+R-1)
        n = npc[idx[:, 1:]].float() if npc is not None else None   # (B,T+R-1,K,F) 各行動の「あと」の住民
        with torch.autocast(**AMP):
            pred = model(z[:, :T], a[:, :T], None if n is None else n[:, :T])
            loss = F.mse_loss(pred.float(), z[:, 1:T + 1])          # 1 歩先 (本物の過去から)
            seq = list(z[:, :T].unbind(1)) + [pred[:, -1].float()]
            extra = 0.0
            for r in range(1, rollout):                             # 2 歩先から: 自分の予測を入力にする
                t = len(seq) - 1; lo = max(0, t - H + 1)
                nxt = model(torch.stack(seq[lo:], 1), a[:, lo:t + 1], None if n is None else n[:, lo:t + 1])[:, -1].float()
                extra = extra + F.mse_loss(nxt, z[:, t + 1])
                seq.append(nxt)
        total = (loss + extra) / rollout
        opt.zero_grad(set_to_none=True); total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); losses.append(loss.item())
        if rollout > 1: far.append(float(extra) / (rollout - 1))
        if (step + 1) % log_every == 0 or step + 1 == steps:
            msg = f"step {step+1:6d}  1 歩先 {np.mean(losses[-log_every:]):.4f}"
            if far: msg += f"  2〜{rollout} 歩先 {np.mean(far[-log_every:]):.4f}"
            print(f"{msg}  ({eta(t0, step + 1, steps)})")
    model.eval()
    return losses


class Perceptual(nn.Module):
    """見た目の近さ: 2 枚の絵を ImageNet で学習済みの VGG16 に通し、途中の特徴 (輪郭・模様) を比べる。
    LPIPS と同じ考え方 (層ごとの重み付けの学習は無し)。復元器の学習にだけ使う。"""
    def __init__(self):
        super().__init__()
        from torchvision.models import vgg16, VGG16_Weights
        self.f = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:16].eval().requires_grad_(False)
        self.taps = {3, 8, 15}                          # relu1_2, relu2_2, relu3_3

    def forward(self, y, x):
        h = (torch.cat([y, x]) - IMNET_MEAN) / IMNET_STD
        loss = 0.0
        for k, layer in enumerate(self.f):
            h = layer(h)
            if k in self.taps:
                a, b = F.normalize(h.float(), dim=1).chunk(2)
                loss = loss + ((a - b) ** 2).sum(1).mean()
        return loss


class PatchDisc(nn.Module):
    """本物らしさの判定役 (PatchGAN)。画像の小さな区画ごとに「本物の描画か、復元器の絵か」を当てる。
    復元器は、これをだませる絵 (くっきりした輪郭や模様のある絵) を出すように学習する"""
    def __init__(self, ch=64):
        super().__init__()
        layers, c = [nn.Conv2d(3, ch, 4, 2, 1), nn.LeakyReLU(0.2)], ch
        for _ in range(2):
            layers += [nn.Conv2d(c, 2 * c, 4, 2, 1), nn.GroupNorm(8, 2 * c), nn.LeakyReLU(0.2)]
            c *= 2
        layers += [nn.Conv2d(c, 1, 3, 1, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x * 2 - 1)


def frame_starts(data):
    """各フレームが入っているエピソードの最初のフレームの番号 (手がかりの画像を同じエピソードから選ぶため)"""
    return torch.repeat_interleave(torch.tensor(data["ep_start"]), torch.tensor(data["ep_len"]))


def thumbs_at(data, idx):
    """縮小画像を番号で取り出す (CPU に置いてあっても、GPU にあってもよい)。返り値は元と同じ場所の uint8 (B,H,W,3)"""
    return data["thumbs"][idx.to(data["thumbs"].device)]


def to_gpu_image(t):
    """uint8 (B,H,W,3) → GPU の float (B,3,H,W) [0,1]。CPU にあるときは、ピン留めしたメモリから計算と並べて送る"""
    if t.device.type == "cpu" and DEVICE == "cuda":
        t = t.pin_memory().to(DEVICE, non_blocking=True)
    return t.to(DEVICE).permute(0, 3, 1, 2).float() / 255


def train_decoder(dec, data, steps, batch=64, lr=1e-3, perceptual=0.0, gan=0.0, noise=0.0, imagined=0.0, ema=0.0,
                  wm=None, log_every=500):
    """特徴 → 画像 を学習する。data: collect() の返り値 (feats は GPU、thumbs は CPU でも GPU でもよい)。

    損失 = 画素の差 (L1) + perceptual × 見た目の近さ + gan × 本物らしさ。L1 だけだと、迷ったところを平均した
    「ぼやけた絵」が得になる。見た目の近さと本物らしさの損失で、輪郭や模様を出させる。
    noise: 特徴に混ぜるノイズの大きさ / imagined: 世界モデルが想像した特徴で学習する例の割合。
           どちらも「想像した特徴は本物より少しずれている」のに慣らすため (想像を見るときの崩れが減る)。
    dec.ctx が True なら、同じエピソードの少し前の本物の画像を手がかりとして渡す (想像では、最後に本物を見たコマを渡す)。
    ema: 重みの移動平均。最後に、平均した重みに置き換える"""
    import copy
    from concurrent.futures import ThreadPoolExecutor
    feats, M = data["feats"], len(data["feats"])
    first = frame_starts(data)
    use_im = imagined > 0 and wm is not None
    n_im = int(round(batch * imagined)) if use_im else 0
    K_IM = 6                                             # 想像させる歩数の上限
    im_starts = valid_starts(data["ep_start"], data["ep_len"], wm.hist + K_IM) if use_im else None

    def make_batch():
        """1 回分の番号を決めて、縮小画像 (答えと手がかり) を取り出しておく。別スレッドで次の分を先に用意する"""
        b = {}
        i = torch.randint(0, M, (batch - n_im,))
        lag = torch.randint(1, 9, (len(i),))
        b["i"], b["ci"] = i, torch.maximum(first[i], i - lag)          # 手がかり: 同じエピソードの 1〜8 歩前
        if n_im:
            s = im_starts[torch.randint(0, len(im_starts), (n_im,))]
            k = random.randint(1, K_IM)
            last = s + wm.hist - 1                                      # 最後に本物を見たコマ
            b.update(s=s, k=k, ti=last + k, ci_im=last)
            tgt, cix = torch.cat([i, last + k]), torch.cat([b["ci"], last])
        else:
            tgt, cix = i, b["ci"]
        b["x"], b["c"] = thumbs_at(data, tgt), thumbs_at(data, cix)
        if b["x"].device.type == "cpu" and DEVICE == "cuda":
            b["x"], b["c"] = b["x"].pin_memory(), b["c"].pin_memory()
        return b

    opt = torch.optim.AdamW(dec.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    pnet = Perceptual().to(DEVICE) if perceptual > 0 else None
    disc = PatchDisc().to(DEVICE) if gan > 0 else None
    d_opt = torch.optim.AdamW(disc.parameters(), lr=2e-4, betas=(0.5, 0.9)) if disc is not None else None
    gan_start = int(steps * 0.5)                        # 前半は L1 と見た目だけで形を覚えさせ、後半から判定役を入れる
    last_w = dec.head[-1].weight if hasattr(dec, "head") else dec.net[-1].weight   # 釣り合いを測る最後の層
    ema_dec = copy.deepcopy(dec).eval().requires_grad_(False) if ema > 0 else None
    dec.train(); logs = {"L1": [], "見た目": [], "本物らしさ": []}; t0 = time.time()
    pool = ThreadPoolExecutor(1); nxt = pool.submit(make_batch)
    for step in range(steps):
        b = nxt.result(); nxt = pool.submit(make_batch)
        x, c = to_gpu_image(b["x"]), to_gpu_image(b["c"])
        z = feats[b["i"].to(feats.device)].float()
        if noise > 0:                                   # 半分の例の特徴にノイズを混ぜる
            keep = (torch.rand(len(z), 1, 1, device=z.device) < 0.5).float()
            z = z + noise * keep * torch.randn_like(z)
        if n_im:                                        # 世界モデルで k 歩想像した特徴 (答えはその k 歩先の本物の画像)
            s = b["s"].to(feats.device)
            z_ctx = feats[s[:, None] + torch.arange(wm.hist, device=feats.device)].float()
            a_seq = data["acts"][s[:, None] + torch.arange(wm.hist - 1 + b["k"], device=feats.device)]
            n_seq = (data["npc"][s[:, None] + 1 + torch.arange(wm.hist - 1 + b["k"], device=feats.device)].float()
                     if data.get("npc") is not None else None)
            z = torch.cat([z, imagine(wm, z_ctx, a_seq, n_seq)[:, -1]])
        if dec.ctx:                                     # 15% は手がかりなし (なくても描けるように)
            c = c * (torch.rand(len(c), 1, 1, 1, device=c.device) > 0.15).float()
        with torch.autocast(**AMP):
            y = dec(z, c) if dec.ctx else dec(z)
            l1 = F.l1_loss(y.float(), x)
            pl = pnet(y, x) if pnet is not None else torch.zeros((), device=x.device)
            g_loss = -disc(y).float().mean() if disc is not None and step >= gan_start else torch.zeros((), device=x.device)
        rec = l1 + perceptual * pl
        if g_loss.requires_grad:
            # 本物らしさの重みは毎回自動で決める (VQGAN と同じ)。最後の層で「画素・見た目の損失」と「本物らしさ」の
            # 勾配の大きさを比べ、本物らしさが勝ちすぎないようにする。固定の重みだと、判定役をだますために
            # 手がかりの画像をそのまま写す (中身を無視して本物っぽい絵を出す) ずるを覚えてしまう
            g_rec = torch.autograd.grad(rec, last_w, retain_graph=True)[0].norm()
            g_gan = torch.autograd.grad(g_loss, last_w, retain_graph=True)[0].norm()
            lam = (g_rec / (g_gan + 1e-4)).clamp(0, 1e4).detach()
        else:
            lam = 1.0
        loss = rec + gan * lam * g_loss
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if disc is not None and step >= gan_start - 500:   # 判定役は少し早めから鍛えておく
            with torch.autocast(**AMP):
                d_loss = F.relu(1 - disc(x)).float().mean() + F.relu(1 + disc(y.detach())).float().mean()
            d_opt.zero_grad(set_to_none=True); d_loss.backward(); d_opt.step()
        if ema_dec is not None:
            with torch.no_grad():
                d = min(ema, (1 + step) / (10 + step))   # 最初は平均を短く (学習の初めの未熟な重みを引きずらない)
                for pe, p in zip(ema_dec.parameters(), dec.parameters()):
                    pe.lerp_(p, 1 - d)
                for be, b_ in zip(ema_dec.buffers(), dec.buffers()):
                    be.copy_(b_)
        logs["L1"].append(l1.item()); logs["見た目"].append(pl.item()); logs["本物らしさ"].append(g_loss.item())
        if (step + 1) % log_every == 0 or step + 1 == steps:
            msg = f"decoder step {step+1:6d}  L1 {np.mean(logs['L1'][-log_every:]):.4f}"
            if pnet is not None: msg += f"  見た目 {np.mean(logs['見た目'][-log_every:]):.4f}"
            if disc is not None and step >= gan_start: msg += f"  本物らしさ {np.mean(logs['本物らしさ'][-log_every:]):+.3f}"
            print(f"{msg}  ({eta(t0, step + 1, steps)})")
    pool.shutdown()
    if ema_dec is not None:
        dec.load_state_dict(ema_dec.state_dict())
    dec.eval()


def ctx_batch(ctx, n):
    """手がかりの画像 (H,W,3) [0,1] の numpy / uint8 のテンソル、またはそれを n 枚並べたもの → GPU の (n,3,H,W)"""
    if ctx is None:
        return None
    t = torch.as_tensor(np.asarray(ctx.cpu() if torch.is_tensor(ctx) else ctx))
    t = t.float() / 255 if t.dtype == torch.uint8 else t.float()
    if t.dim() == 3:
        t = t[None].expand(n, -1, -1, -1)
    return t.permute(0, 3, 1, 2).to(DEVICE)


@torch.no_grad()
def decode(dec, z, ctx=None, chunk=256):
    """(…,N,D) → (…,H,W,3) numpy [0,1]。ctx: 手がかりの画像 (1 枚か、z と同じ数)。chunk 枚ずつ戻す (メモリを抑える)"""
    sh = z.shape[:-2]
    zf = z.reshape(-1, *z.shape[-2:]).float()
    c = ctx_batch(ctx, len(zf)) if getattr(dec, "ctx", False) else None
    out = []
    for k in range(0, len(zf), chunk):
        with torch.autocast(**AMP):
            img = dec(zf[k:k + chunk], c[k:k + chunk]) if c is not None else dec(zf[k:k + chunk])
        out.append(img.float().permute(0, 2, 3, 1).cpu())
    img = torch.cat(out)
    return img.reshape(*sh, *img.shape[1:]).numpy()

In [ ]:
def show_rows(rows, cols_title=None, size=1.6, title=None):
    """rows: {行の名前: [画像 (H,W,3), ...]} を表にして並べる"""
    names = list(rows); n = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(len(names), n, figsize=(size * n * 1.2, size * len(names)), squeeze=False)
    for r, name in enumerate(names):
        for c in range(n):
            a = ax[r, c]; a.axis("off")
            if c < len(rows[name]):
                a.imshow(np.clip(rows[name][c], 0, 1))
            if c == 0:
                a.text(-0.08, 0.5, name, transform=a.transAxes, ha="right", va="center", fontsize=10)
            if r == 0 and cols_title is not None:
                a.set_title(cols_title[c], fontsize=9)
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()


def animate_rows(rows, fps=6, size=2.2):
    """rows: {名前: (T,H,W,3)} を横に並べた動画 (Colab の中で再生できる HTML)"""
    names = list(rows); T = min(len(v) for v in rows.values())
    fig, ax = plt.subplots(1, len(names), figsize=(size * len(names) * 1.2, size + 0.4), squeeze=False)
    ims = []
    for a, name in zip(ax[0], names):
        a.axis("off"); a.set_title(name, fontsize=10)
        ims.append(a.imshow(np.clip(rows[name][0], 0, 1)))
    def upd(t):
        for im, name in zip(ims, names):
            im.set_data(np.clip(rows[name][t], 0, 1))
        return ims
    anim = animation.FuncAnimation(fig, upd, frames=T, interval=1000 / fps)
    plt.close(fig)
    return HTML(anim.to_jshtml())

## データを作る (地図 → 目線の描画 → DINOv2 の特徴)

In [ ]:
# ── 地図: persona の学習ノートブックの make_map_organic と同じ作り方 ──
OTHER, ROAD, BUILDING, TREE = 0, 1, 2, 3
from collections import deque

def make_map_organic(size, seed):
    rng = random.Random(seed)
    grid = np.full((size, size), OTHER, dtype=np.int32)
    rows = cols = list(range(0, size, 4))
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD
    for ri in range(len(rows) - 1):
        for ci in range(len(cols) - 1):
            cells = [(r, c) for r in range(rows[ri] + 1, rows[ri + 1]) for c in range(cols[ci] + 1, cols[ci + 1])]
            if not cells: continue
            b = rng.choice(cells); grid[b] = BUILDING
            for rc in cells:
                if rc == b: continue
                v = rng.random()
                grid[rc] = TREE if v < 0.25 else BUILDING if v < 0.45 else grid[rc]
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD

    def connected(g):
        cells = list(zip(*np.where(g == ROAD)))
        seen = {cells[0]}; q = deque([cells[0]])
        while q:
            r, c = q.popleft()
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                n = (r + dr, c + dc)
                if 0 <= n[0] < size and 0 <= n[1] < size and n not in seen and g[n] == ROAD:
                    seen.add(n); q.append(n)
        return len(seen) == len(cells)

    cand = [(r, c) for r, c in zip(*np.where(grid == ROAD)) if not (r in rows and c in cols)]
    rng.shuffle(cand)
    quota = int(len(cand) * (0.25 + rng.random() * 0.25)); removed = 0
    for rc in cand:
        if removed >= quota: break
        grid[rc] = OTHER
        if connected(grid):
            grid[rc] = TREE if rng.random() < 0.4 else OTHER; removed += 1
        else:
            grid[rc] = ROAD
    return grid


# ── 建物の種類と高さ: mesa-clouds-renderer の BLDG_TYPES (server.js) と同じ ──
#   (名前 = textures/v4/<名前>.jpg, 敷地 1=1x1 / 2=2x2 マス, 高さ [マス])
BLDG_TYPES = [
    ("kiosk", 1, 0.7), ("conbini", 1, 0.9), ("pharmacy", 1, 0.9), ("cafe", 1, 1.1), ("gyudon", 1, 1.1),
    ("ramen", 1, 1.1), ("bento", 1, 1.1), ("shop", 1, 1.4), ("house", 1, 1.4), ("post", 1, 1.4),
    ("bank", 1, 1.7), ("apartment", 1, 2.1), ("hotel", 1, 2.1), ("office", 1, 2.6), ("tower", 1, 3.3),
    ("police", 1, 1.7), ("elementary", 1, 1.1), ("junior", 1, 1.4), ("warehouse", 1, 1.4),
    ("supermarket", 2, 1.1), ("temple", 2, 1.1), ("school", 2, 1.4), ("station", 2, 1.4), ("library", 2, 1.4),
    ("hospital", 2, 1.7), ("cityhall", 2, 1.7), ("museum", 2, 1.7), ("stadium", 2, 2.1), ("mall", 2, 2.6),
    ("high", 2, 1.7), ("university", 2, 2.1),
]
SMALL = [i for i, t in enumerate(BLDG_TYPES) if t[1] == 1]
BIG = [i for i, t in enumerate(BLDG_TYPES) if t[1] == 2]
P_BIG = 0.3          # 街区 (道に囲まれた 3x3) に 2x2 の大きな建物を置く割合

def bldg_height(t, r, c):
    """業種の高さ × 座標から決まる倍率 (±12% を 5 段階)。renderer と同じく、同じ業種が定規で揃わないように"""
    k = ((r * 73856093) ^ (c * 19349663)) % 5
    return BLDG_TYPES[t][2] * (0.88 + 0.06 * k)

def place_buildings(grid, seed):
    """建物のマスに業種を割り当てる。返り値: 地図 (2x2 を足したもの) と、マスごとの
    業種 bt (建物でなければ -1)・高さ bh・建物の左上のマス org・敷地の幅 fp"""
    rng = np.random.default_rng(seed)
    grid = grid.copy(); S = grid.shape[0]
    bt = np.full(grid.shape, -1, np.int64); bh = np.zeros(grid.shape, np.float32)
    org = np.zeros(grid.shape + (2,), np.int64); fp = np.ones(grid.shape, np.int64)
    # 1) いくつかの街区に 2x2 の大きな建物 (スーパー・学校・病院 …)。街区の中なので道は塞がない
    for r0 in range(1, S - 3, 4):
        for c0 in range(1, S - 3, 4):
            if rng.random() >= P_BIG: continue
            r, c = r0 + int(rng.integers(0, 2)), c0 + int(rng.integers(0, 2))
            t = int(rng.choice(BIG)); sl = (slice(r, r + 2), slice(c, c + 2))
            grid[sl] = BUILDING; bt[sl] = t; org[sl] = (r, c); fp[sl] = 2; bh[sl] = bldg_height(t, r, c)
    # 2) 残りの建物のマスは 1x1 の店や家
    for r, c in zip(*np.where((grid == BUILDING) & (bt < 0))):
        t = int(rng.choice(SMALL)); bt[r, c] = t; org[r, c] = (r, c); bh[r, c] = bldg_height(t, r, c)
    return grid, bt, bh, org, fp


# ── 地面: renderer の rebuildGround と同じ貼り分け ──
#   道 → アスファルト + 路面標示 (歩道・白線・横断歩道)、建物の足元 → 舗装、木の足元・空き地 → 芝生、
#   踏み固められた空き地 → 土 (renderer は人の通った回数で決める。ここでは座標から 3 割)
G_ASPHALT, G_GRASS, G_DIRT = 0, 1, 2
SLOT_TWOLANE, SLOT_ONEWAY, SLOT_PATH = 0, 16, 32   # 路面標示アトラスの枠 (roads.js と同じ)

def ground_layout(grid):
    S = grid.shape[0]
    gk = np.full(grid.shape, G_GRASS, np.int64); slot = np.full(grid.shape, -1, np.int64)
    for r in range(S):
        for c in range(S):
            t = grid[r, c]
            if t in (ROAD, BUILDING): gk[r, c] = G_ASPHALT
            elif t == OTHER and ((r * 92837111) ^ (c * 689287499)) % 10 < 3: gk[r, c] = G_DIRT
            if t != ROAD: continue
            R = lambda rr, cc: 0 <= rr < S and 0 <= cc < S and grid[rr, cc] == ROAD
            mask = R(r - 1, c) | R(r, c + 1) << 1 | R(r + 1, c) << 2 | R(r, c - 1) << 3   # N|E|S|W
            slot[r, c] = SLOT_PATH if bin(mask).count("1") <= 1 else SLOT_ONEWAY + mask
    return gk, slot


t0 = time.time()
MAP_SEEDS = list(range(N_MAPS_TRAIN)) + [10_000 + s for s in range(N_MAPS_TEST)]   # 後ろの N_MAPS_TEST 枚は評価専用
MAPS, _bt, _bh, _org, _fp, _gk, _slot = [], [], [], [], [], [], []
for s in MAP_SEEDS:
    g, bt, bh, org, fp = place_buildings(make_map_organic(GRID, s), s)
    gk, slot = ground_layout(g)
    for lst, v in zip((MAPS, _bt, _bh, _org, _fp, _gk, _slot), (g, bt, bh, org, fp, gk, slot)): lst.append(v)
_T = lambda a, **k: torch.tensor(np.stack(a), device=DEVICE, **k)
BT, BH, BORG, BFP = _T(_bt), _T(_bh, dtype=torch.float32), _T(_org), _T(_fp)   # BT[地図, x, y] = 業種 (建物でなければ -1)
GK, RSLOT = _T(_gk), _T(_slot)
TREEM = _T([g == TREE for g in MAPS])                                        # 木のマス (通れるが、目には見える)
print(f"地図 {len(MAPS)} 枚 ({time.time()-t0:.0f}s)")

cmap = plt.matplotlib.colors.ListedColormap(["#1a3a6a", "#d0d0c8", "#c42020", "#256b28"])
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for j, k in enumerate([0, 1, N_MAPS_TRAIN, N_MAPS_TRAIN + 1]):
    ax[0, j].imshow(MAPS[k], cmap=cmap, vmin=0, vmax=3)
    ax[0, j].set_title(("学習用" if k < N_MAPS_TRAIN else "評価用") + f" seed={MAP_SEEDS[k]}", fontsize=9)
    hcm = plt.get_cmap("viridis").copy(); hcm.set_bad("#e4e4e0")          # 建物でないマスは薄い灰色
    im = ax[1, j].imshow(np.ma.masked_where(_bt[k] < 0, _bh[k]), cmap=hcm, vmin=0.5, vmax=3.7)
    ax[1, j].set_title("建物の高さ [マス]", fontsize=9)
for a in ax.flat: a.axis("off")
fig.colorbar(im, ax=ax[1].tolist(), shrink=0.8)
plt.show()

In [ ]:
# ── 目線の描画: mesa-clouds-renderer の 3D 表示に寄せたテクスチャ付き DDA レイキャスタ ──
#   ・建物の壁 = renderer と同じ写真 (textures/v4)。1 枚で建物の正面まるごとの絵なので、壁の高さ全体に貼る
#   ・建物の高さ = 業種ごとの高さ。手前の低い建物の上から、奥の高いビルがのぞいて見える
#   ・地面 = アスファルト + 路面標示 (歩道・白線・横断歩道) / 芝生 / 土。renderer の textures/base と road_atlas
#   ・空 = renderer の昼の空 (地平線が明るく、上ほど少し濃い)
import urllib.request, io
from PIL import Image

RAW = "https://raw.githubusercontent.com/oggata/mesa-clouds-renderer/main/"
def fetch(path, mode="RGB"):
    """renderer の素材を WORK_DIR/mesa_assets/ に落として読む (2 回目からは Drive にあるものを使う)"""
    local = os.path.join(WORK_DIR, "mesa_assets", path)
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        urllib.request.urlretrieve(RAW + path, local + ".part")   # 途中で切れても壊れたファイルが残らないよう、落とし終えてから名前を変える
        os.replace(local + ".part", local)
    return Image.open(local).convert(mode)
def to_t(im, size):
    return torch.tensor(np.asarray(im.resize(size, Image.BILINEAR), np.float32) / 255, device=DEVICE)

TW, TH = 192, 256           # 壁の写真 (縦長の写真なので縦を多めに。近づいてもぼやけない大きさ)
GT = 128                    # 地面の模様 1 マスぶん
AT = 64                     # 路面標示の 1 枠
TEX = torch.stack([to_t(fetch(f"textures/v4/{n}.jpg"), (TW, TH)) for n, _, _ in BLDG_TYPES])   # (業種,TH,TW,3)
_dirt = to_t(fetch("textures/base/other_256.jpg"), (GT, GT)) * torch.tensor([0xb5, 0xb0, 0x8a], device=DEVICE) / 255 * 1.55
_asph = to_t(fetch("textures/base/asphalt_256.jpg"), (GT, GT))
_asph = _asph / _asph.mean((0, 1)) * torch.tensor([0.265, 0.275, 0.295], device=DEVICE)   # 車道の色 (roads.js の FLOOR_RGB.ASPHALT)
GTEX = torch.stack([_asph,
                    to_t(fetch("textures/base/grass_256.jpg"), (GT, GT)), _dirt.clamp(0, 1)])  # G_ASPHALT, G_GRASS, G_DIRT
_atlas = fetch("textures/road/road_atlas.png", "RGBA")   # 8x8 枠 (1 枠 128px、まわり 8px は継ぎ目用の余白)
ATLAS = torch.stack([to_t(_atlas.crop(((s % 8) * 128 + 8, (s // 8) * 128 + 8, (s % 8) * 128 + 120, (s // 8) * 128 + 120)), (AT, AT))
                     for s in range(33)])               # (33,AT,AT,4)。車道の内側は透明 (下のアスファルトが見える)

def make_tree_sprite(n=64, seed=0):
    """木の絵 (RGBA)。幹 + 丸い樹冠。renderer の木と同じく、丸い緑の頭に茶色の幹"""
    g = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:n, 0:n] / n
    crown = ((xx - 0.5) / 0.48) ** 2 + ((yy - 0.36) / 0.36) ** 2 < 1
    trunk = (abs(xx - 0.5) < 0.06) & (yy > 0.55)
    blot = np.kron(g.standard_normal((n // 8, n // 8)), np.ones((8, 8)))      # 葉のまだら (幅の 1/8 のかたまり)
    shade = 0.75 + 0.35 * (0.5 - yy) + 0.10 * blot                             # 上が明るい
    rgb = np.where(crown[..., None], np.array([0x48, 0x90, 0x3f]) / 255 * shade[..., None],
                   np.array([0x7a, 0x52, 0x30]) / 255)
    a = (crown | trunk).astype(np.float32)
    return torch.tensor(np.dstack([rgb.clip(0, 1), a]), device=DEVICE, dtype=torch.float32)
TREE_TEX = make_tree_sprite(128)
TREE_H, TREE_R = 1.0, 0.32  # 木の高さと半径 [マス]

def _rgb(h): return torch.tensor([(h >> 16) & 255, (h >> 8) & 255, h & 255], device=DEVICE, dtype=torch.float32) / 255
SKY_H = _rgb(0xeaf2f7)                                    # 地平線 (renderer の昼の背景色 = フォグの色)
SKY_Z = SKY_H * 0.78 + (_rgb(0x1b3a63) - SKY_H * 0.78) * 0.10   # 天頂 (renderer の空ドームと同じ混ぜ方)
FOV = math.radians(68)      # 視野角 (renderer の一人称 FPV_FOV)
EYE = 0.22                  # 目の高さ [マス] (renderer の住民の目線 0.44 / CELL 2.0)
SSAA = 2                    # 2 倍の大きさで描いて縮める (地面の模様のちらつき止め)
LAYERS = 4                  # 1 列に重ねて描く壁の枚数 (手前の低い建物の奥にある高い建物を見せる)
TLAYERS = 3                 # 1 列に重ねて描く木の枚数
FOG0, FOG1 = 6.0, 30.0      # この距離 [マス] から空の色に溶けはじめ、FOG1 で 7 割

def _fog(col, dist):
    f = ((dist - FOG0) / (FOG1 - FOG0)).clamp(0, 1) * 0.7
    return col * (1 - f[..., None]) + SKY_H * f[..., None]

# ── ほかの住民 (NPC) の絵: 木と同じく、こちらを向いた板 (スプライト)。人ごとにシャツの色を変えて見分けられるようにする ──
PERSON_H, PERSON_W = 0.26, 0.13   # 背の高さと幅 [マス] (目の高さ EYE = 0.22 の人と同じくらい)
PERSON_COLORS = [0xe03c31, 0x2f6fde, 0xf2c230, 0x2fae5a, 0xb04fd6, 0xff8a1f, 0x18b5b5, 0xf06aa8]

def make_person_sprite(shirt, n=64):
    """人の絵 (RGBA)。頭・シャツ・ズボン (2 本の脚)"""
    yy, xx = (np.mgrid[0:n, 0:n // 2] + 0.5) / np.array([n, n // 2])[:, None, None]
    head = ((xx - 0.5) / 0.24) ** 2 + ((yy - 0.11) / 0.10) ** 2 < 1
    body = (abs(xx - 0.5) < 0.30) & (yy > 0.20) & (yy < 0.60)
    legs = (yy >= 0.60) & ((abs(xx - 0.36) < 0.11) | (abs(xx - 0.64) < 0.11))
    col = np.array([(shirt >> 16) & 255, (shirt >> 8) & 255, shirt & 255]) / 255
    rgb = np.where(head[..., None], np.array([0.93, 0.76, 0.62]),
                   np.where(body[..., None], col * (0.85 + 0.25 * (0.6 - yy))[..., None], np.array([0.20, 0.22, 0.30])))
    return np.dstack([rgb.clip(0, 1), (head | body | legs).astype(np.float32)]).astype(np.float32)

def draw_people(img, zbuf, xs, ys, ths, people, FOC, hor, eye, tex, fog):
    """img: (N,H,W,3) / zbuf: (N,H,W) その画素に見えている物までの距離 / people: (N,K,3) 住民の x, y, 向き。
    遠い人から順に描き、壁や木より奥にいる部分は描かない (zbuf と比べる)"""
    N, H, W, _ = img.shape
    K = people.shape[1]
    dirX, dirY = torch.cos(ths)[:, None], torch.sin(ths)[:, None]
    rx, ry = people[..., 0] - xs[:, None], people[..., 1] - ys[:, None]
    dz = rx * dirX + ry * dirY                                  # 前方への距離
    s = -rx * dirY + ry * dirX                                  # 右へのずれ
    order = torch.argsort(dz, dim=1, descending=True)           # 遠い順
    cols = torch.arange(W, device=img.device, dtype=torch.float32) + 0.5
    rows = torch.arange(H, device=img.device, dtype=torch.float32) + 0.5
    T = tex.shape
    for j in range(K):
        k = order[:, j]
        d = dz.gather(1, k[:, None])[:, 0]; sk = s.gather(1, k[:, None])[:, 0]
        dd = d.clamp(min=1e-3)
        cx = W / 2 + FOC * sk / dd
        hw = FOC * PERSON_W / 2 / dd
        top = hor - FOC * (PERSON_H - eye) / dd; bot = hor + FOC * eye / dd
        u = (cols[None] - (cx - hw)[:, None]) / (2 * hw)[:, None]                 # (N,W)
        v = (rows[None] - top[:, None]) / (bot - top)[:, None]                    # (N,H)
        inside = ((d > 0.15)[:, None, None] & (u >= 0)[:, None, :] & (u < 1)[:, None, :]
                  & (v >= 0)[:, :, None] & (v < 1)[:, :, None])
        ui = (u.clamp(0, 0.999) * T[2]).long()[:, None, :].expand(N, H, W)
        vi = (v.clamp(0, 0.999) * T[1]).long()[:, :, None].expand(N, H, W)
        tc = tex[(k % T[0])[:, None, None].expand(N, H, W), vi, ui]               # (N,H,W,4)
        on = inside & (tc[..., 3] > 0.5) & (d[:, None, None] < zbuf)
        img = torch.where(on[..., None], fog(tc[..., :3], d[:, None, None].expand(N, H, W)), img)
        zbuf = torch.where(on, d[:, None, None].expand(N, H, W), zbuf)
    return img
PERSON_TEX = torch.tensor(np.stack([make_person_sprite(c) for c in PERSON_COLORS]), device=DEVICE)   # (色,64,32,4)

@torch.no_grad()
def _render(xs, ys, ths, mid, H, W, people=None, MAXSTEPS=64):
    """(N,) → (N,H,W,3)。列 0 が th - FOV/2 を見る = th を減らすと画面の左を向く (既存方策の turn_left と同じ)"""
    N = xs.shape[0]
    cam = 2 * torch.arange(W, device=DEVICE, dtype=torch.float32) / W - 1
    dirX, dirY = torch.cos(ths), torch.sin(ths)
    pl = math.tan(FOV / 2); planeX, planeY = -dirY * pl, dirX * pl
    FOC = (W / 2) / pl                      # 焦点距離 [px]。縦も横と同じ倍率で写す (縦に伸びない)
    hor = H / 2
    yc = torch.arange(H, device=DEVICE, dtype=torch.float32) + 0.5   # 画素の中心の行
    rdx = dirX[:, None] + planeX[:, None] * cam[None]; rdy = dirY[:, None] + planeY[:, None] * cam[None]
    m3 = mid[:, None, None]

    # ── 空 ──
    t = ((hor - yc) / hor).clamp(0, 1) ** 0.65
    img = (SKY_H + (SKY_Z - SKY_H) * t[:, None])[None, :, None, :].expand(N, H, W, 3).clone()

    # ── 地面 (床キャスト): 地平線より下の行は、その行が指す地面の 1 点の色 ──
    fr = yc > hor
    dist = FOC * EYE / (yc[fr] - hor)                                    # (Hf,) その行の地面までの距離
    fx = xs[:, None, None] + rdx[:, None, :] * dist[None, :, None]       # (N,Hf,W)
    fy = ys[:, None, None] + rdy[:, None, :] * dist[None, :, None]
    r, c = torch.floor(fx).long(), torch.floor(fy).long()
    inb = (r >= 0) & (r < GRID) & (c >= 0) & (c < GRID)
    rr, cc = r.clamp(0, GRID - 1), c.clamp(0, GRID - 1)
    fu, fv = fy - torch.floor(fy), fx - torch.floor(fx)                  # u = 列 (東) 方向, v = 行 (南) 方向
    base = GTEX[GK[m3, rr, cc], (fv * GT).long().clamp(0, GT - 1), (fu * GT).long().clamp(0, GT - 1)]
    slot = RSLOT[m3, rr, cc]
    mk = ATLAS[slot.clamp(min=0), (fv * AT).long().clamp(0, AT - 1), (fu * AT).long().clamp(0, AT - 1)]
    a = mk[..., 3:] * (slot >= 0)[..., None]
    floor = _fog(base * (1 - a) + mk[..., :3] * a, dist[None, :, None].expand_as(fx))
    img[:, fr] = torch.where(inb[..., None], floor, SKY_H)               # 地図の外は何も無い (renderer と同じく背景色)

    # ── 壁: DDA でレイを進め、「手前の壁の上端より上に見える壁」を最大 LAYERS 枚まで拾う ──
    mapX = torch.floor(xs).long()[:, None].expand(N, W).clone()
    mapY = torch.floor(ys).long()[:, None].expand(N, W).clone()
    ddx = torch.where(rdx == 0, torch.full_like(rdx, 1e30), (1 / rdx).abs())
    ddy = torch.where(rdy == 0, torch.full_like(rdy, 1e30), (1 / rdy).abs())
    stepX = torch.where(rdx < 0, -1, 1); stepY = torch.where(rdy < 0, -1, 1)
    pX, pY = xs[:, None], ys[:, None]
    sdx = torch.where(rdx < 0, (pX - mapX) * ddx, (mapX + 1 - pX) * ddx)
    sdy = torch.where(rdy < 0, (pY - mapY) * ddy, (mapY + 1 - pY) * ddy)
    side = torch.zeros((N, W), dtype=torch.long, device=DEVICE)
    active = torch.ones((N, W), dtype=torch.bool, device=DEVICE)
    occ = torch.full((N, W), float(H), device=DEVICE)        # この行より下は手前の壁で埋まっている
    cnt = torch.zeros((N, W), dtype=torch.long, device=DEVICE)
    L = {k: torch.zeros((LAYERS, N, W), device=DEVICE) for k in ("perp", "top", "bot", "u", "occ", "side")}
    Lt = torch.full((LAYERS, N, W), -1, dtype=torch.long, device=DEVICE)
    tcnt = torch.zeros((N, W), dtype=torch.long, device=DEVICE)
    TLr = {k: torch.zeros((TLAYERS, N, W), device=DEVICE) for k in ("dz", "u", "h", "occ")}
    TLv = torch.zeros((TLAYERS, N, W), dtype=torch.bool, device=DEVICE)
    m = mid[:, None].expand(N, W)
    for _ in range(MAXSTEPS):
        xlt = sdx < sdy; sx = active & xlt; sy = active & ~xlt
        sdx = torch.where(sx, sdx + ddx, sdx); mapX = torch.where(sx, mapX + stepX, mapX); side = torch.where(sx, 0, side)
        sdy = torch.where(sy, sdy + ddy, sdy); mapY = torch.where(sy, mapY + stepY, mapY); side = torch.where(sy, 1, side)
        inb = (mapX >= 0) & (mapX < GRID) & (mapY >= 0) & (mapY < GRID)
        mx, my = mapX.clamp(0, GRID - 1), mapY.clamp(0, GRID - 1)
        t = BT[m, mx, my]
        perp = torch.where(side == 0, sdx - ddx, sdy - ddy).clamp(min=1e-4)
        top = hor - FOC * (BH[m, mx, my] - EYE) / perp
        vis = active & inb & (t >= 0) & (top < occ) & (cnt < LAYERS)
        # 壁の横方向の位置 u (0..1)。2x2 の建物は 2 マスにまたがって 1 枚の写真を貼る
        o, fpw = BORG[m, mx, my], BFP[m, mx, my]
        u = torch.where(side == 0, pY + perp * rdy - o[..., 1], pX + perp * rdx - o[..., 0]) / fpw
        flip = ((side == 0) & (rdx > 0)) | ((side == 1) & (rdy < 0)); u = torch.where(flip, 1 - u, u)
        # 木: マスの中心に立つ、こちらを向いた板。通り抜けられるので壁と違って奥を隠さない (絵の透明部分は透ける)
        dz = (mx + 0.5 - pX) * dirX[:, None] + (my + 0.5 - pY) * dirY[:, None]            # 前方への距離
        s_ = (pX + rdx * dz - mx - 0.5) * -dirY[:, None] + (pY + rdy * dz - my - 0.5) * dirX[:, None]  # 横のずれ
        th_ = TREE_H * (0.85 + 0.075 * (((mx * 73856093) ^ (my * 19349663)) % 5))
        tvis = (active & inb & TREEM[m, mx, my] & (dz > 0.3) & (s_.abs() < TREE_R) & (tcnt < TLAYERS)
                & (hor - FOC * (th_ - EYE) / dz.clamp(min=1e-4) < occ))
        for k in range(TLAYERS):
            sel = tvis & (tcnt == k)
            TLv[k] |= sel
            for name, v in (("dz", dz), ("u", s_ / (2 * TREE_R) + 0.5), ("h", th_.float()), ("occ", occ)):
                TLr[name][k] = torch.where(sel, v, TLr[name][k])
        tcnt = tcnt + tvis.long()
        for k in range(LAYERS):
            sel = vis & (cnt == k)
            Lt[k] = torch.where(sel, t, Lt[k])
            for name, v in (("perp", perp), ("top", top), ("bot", hor + FOC * EYE / perp), ("u", u),
                            ("occ", occ), ("side", side.float())):
                L[name][k] = torch.where(sel, v, L[name][k])
        occ = torch.where(vis, top, occ); cnt = cnt + vis.long()
        active = active & inb & (occ > 0) & (cnt < LAYERS)
        if not active.any(): break

    rows = yc[None, :, None]
    zbuf = torch.full((N, H, W), float("inf"), device=DEVICE)             # その画素に見えている物までの距離 (住民の隠れに使う)
    for k in range(LAYERS):
        tk = Lt[k]
        top, bot = L["top"][k][:, None, :], L["bot"][k][:, None, :]
        inwall = (tk >= 0)[:, None, :] & (rows >= top) & (rows < bot) & (rows < L["occ"][k][:, None, :])
        ty = ((rows - top) / (bot - top)).clamp(0, 0.999) * TH          # 写真は壁の高さ全体に貼る
        tx = (L["u"][k].clamp(0, 0.999) * TW).long()
        col = TEX[tk.clamp(min=0)[:, None, :].expand(N, H, W), ty.long(), tx[:, None, :].expand(N, H, W)]
        shade = torch.where(L["side"][k] > 0, 0.82, 1.0)                   # 面の向きで明るさを変える (日なた / 日かげ)
        col = _fog(col * shade[:, None, :, None], L["perp"][k][:, None, :].expand(N, H, W))
        img = torch.where(inwall[..., None], col, img)
        zbuf = torch.where(inwall, L["perp"][k][:, None, :].expand(N, H, W), zbuf)
    for k in reversed(range(TLAYERS)):                                     # 木は奥から順に重ねる
        dz = TLr["dz"][k].clamp(min=1e-4)[:, None, :]
        top = hor - FOC * (TLr["h"][k][:, None, :] - EYE) / dz; bot = hor + FOC * EYE / dz
        ty = ((rows - top) / (bot - top)).clamp(0, 0.999) * TREE_TEX.shape[0]
        tx = (TLr["u"][k].clamp(0, 0.999) * TREE_TEX.shape[1]).long()[:, None, :].expand(N, H, W)
        tc = TREE_TEX[ty.long().expand(N, H, W), tx]
        on = TLv[k][:, None, :] & (rows >= top) & (rows < bot) & (rows < TLr["occ"][k][:, None, :]) & (tc[..., 3] > 0.5)
        img = torch.where(on[..., None], _fog(tc[..., :3], dz.expand(N, H, W)), img)
        zbuf = torch.where(on, dz.expand(N, H, W), zbuf)
    if people is not None:                                                 # ほかの住民 (遠い順に、壁や木に隠れる)
        img = draw_people(img, zbuf, xs, ys, ths, people, FOC, hor, EYE, PERSON_TEX, _fog)
    return img

@torch.no_grad()
def render(xs, ys, ths, mid, H=IMG, W=IMG, chunk=32, people=None):
    """xs, ys, ths, mid (地図の番号): (N,) → 目線の画像 (N,3,H,W) [0,1]。people: (N,K,3) ほかの住民の x, y, 向き"""
    out = []
    for i in range(0, xs.shape[0], chunk):
        sl = slice(i, i + chunk)
        im = _render(xs[sl], ys[sl], ths[sl], mid[sl], H * SSAA, W * SSAA,
                     people=None if people is None else people[sl]).permute(0, 3, 1, 2)
        out.append(F.avg_pool2d(im, SSAA) if SSAA > 1 else im)
    return torch.cat(out).clamp(0, 1).contiguous()


# ── 動き方: 前進・左・右は MOVE だけ進み、左右は ROT だけ曲がりながら進む (MOVE_MODE=bc と同じ運動学)。
#    後退は向きを変えずに MOVE だけ下がる。建物のマスや地図の外へは進めず、その場に留まる (ぶつかる)。
@torch.no_grad()
def step_env(x, y, th, a, mid):
    th = th + torch.where(a == 1, -ROT, torch.where(a == 2, ROT, 0.0))
    sgn = torch.where(a == 3, -1.0, 1.0)
    nx, ny = x + sgn * MOVE * torch.cos(th), y + sgn * MOVE * torch.sin(th)
    inb = (nx > 0.05) & (nx < GRID - 0.05) & (ny > 0.05) & (ny < GRID - 0.05)
    free = BT[mid, nx.long().clamp(0, GRID - 1), ny.long().clamp(0, GRID - 1)] < 0
    ok = inb & free
    return torch.where(ok, nx, x), torch.where(ok, ny, y), th, ok


def random_starts(mid):
    """建物でないマスにランダムに置く"""
    xs, ys = [], []
    for m in mid.tolist():
        free = np.argwhere(MAPS[m] != BUILDING)
        r, c = free[np.random.randint(len(free))]
        xs.append(r + np.random.uniform(0.2, 0.8)); ys.append(c + np.random.uniform(0.2, 0.8))
    th = torch.rand(len(xs), device=DEVICE) * 2 * math.pi
    return torch.tensor(xs, device=DEVICE, dtype=torch.float32), torch.tensor(ys, device=DEVICE, dtype=torch.float32), th


class Explorer:
    """データ集め用の行動: 同じ行動をしばらく続け、ときどき変える (前進 4.5 割・左右 2 割ずつ・後退 1.5 割)"""
    def __init__(self, n):
        self.a = self.sample(n)
    def sample(self, n):
        return torch.multinomial(torch.tensor([0.45, 0.2, 0.2, 0.15], device=DEVICE), n, replacement=True)
    def __call__(self):
        change = torch.rand(len(self.a), device=DEVICE) < 0.25
        self.a = torch.where(change, self.sample(len(self.a)), self.a)
        return self.a


# 試しに 4 人分の目線を描く (道の上から、道の続く向きを見る)
def road_starts(mid):
    xs, ys, ths = [], [], []
    for m in mid.tolist():
        g = MAPS[m]; cand = np.argwhere(g == ROAD)
        r, c = cand[np.random.randint(len(cand))]
        dirs = [k for k, (dr, dc) in enumerate(((1, 0), (0, 1), (-1, 0), (0, -1)))
                if 0 <= r + dr < GRID and 0 <= c + dc < GRID and g[r + dr, c + dc] == ROAD] or [0]
        xs.append(r + 0.5); ys.append(c + 0.5); ths.append(np.random.choice(dirs) * math.pi / 2 + np.random.uniform(-0.3, 0.3))
    f = lambda v: torch.tensor(v, device=DEVICE, dtype=torch.float32)
    return f(xs), f(ys), f(ths)

mid = torch.tensor([0, 0, 1, N_MAPS_TRAIN], device=DEVICE)
x, y, th = road_starts(mid)
show_rows({"目線": list(render(x, y, th, mid).permute(0, 2, 3, 1).cpu().numpy())},
          [f"地図 {m}" for m in mid.tolist()], size=2.6)

In [ ]:
# ── ほかの住民 (NPC) の動き: 道のマスの真ん中を、マスからマスへまっすぐ歩く。
#    次のマスに着くたびに行き先を選び直す (まっすぐが選ばれやすく、引き返すのは行き止まりのときだけ)。
#    位置はこのシミュレーターが決める「本当の状態」。世界モデルには、これをプレイヤーから見た数の並び (テンソル) で渡す
DIRS4 = torch.tensor([[1, 0], [0, 1], [-1, 0], [0, -1]], device=DEVICE)   # 向きの番号 0..3 = 向き 0, 90, 180, 270 度
NPC_SPEED = 0.08     # 1 歩で進むマス (プレイヤーの MOVE = 0.2 の半分弱。人はゆっくり歩く)
NPC_RANGE = 8.0      # これより遠い住民は「いない」として渡す [マス]
NPC_F = 6            # 1 人あたりの数: 前への距離・右への距離 (どちらも ÷NPC_RANGE)・向きの sin, cos・距離・いるか (0/1)
ROADM = torch.tensor(np.stack([g == ROAD for g in MAPS]), device=DEVICE)   # (地図, x, y) 道のマス

class People:
    """K 人の住民を、地図 mid (N,) ごとに歩かせる。pose() で (N,K,3) の x, y, 向き"""
    def __init__(self, mid, k, seed=None):
        g = torch.Generator(device="cpu")
        g.manual_seed(int(seed) if seed is not None else random.randrange(1 << 30))
        self.mid, self.k, self.g = mid, k, g
        r, c = [], []
        for m in mid.tolist():
            cand = torch.nonzero(ROADM[m].cpu())
            pick = cand[torch.randint(len(cand), (k,), generator=g)]
            r.append(pick[:, 0]); c.append(pick[:, 1])
        self.r, self.c = torch.stack(r).to(DEVICE), torch.stack(c).to(DEVICE)
        self.d = self.choose(self.r, self.c, None)
        self.p = torch.rand(self.r.shape, generator=g).to(DEVICE)       # 次のマスまでのどこにいるか (0..1)

    def options(self, r, c):
        """(N,K) の各マスから、道が続いている向き (N,K,4)"""
        nr, nc = r[..., None] + DIRS4[:, 0], c[..., None] + DIRS4[:, 1]
        inb = (nr >= 0) & (nr < GRID) & (nc >= 0) & (nc < GRID)
        m = self.mid[:, None, None].expand_as(nr)
        return inb & ROADM[m, nr.clamp(0, GRID - 1), nc.clamp(0, GRID - 1)]

    def choose(self, r, c, d):
        w = self.options(r, c).float()
        if d is not None:
            w = w * (1 + 3 * F.one_hot(d, 4)) * (1 - 0.97 * F.one_hot((d + 2) % 4, 4))   # まっすぐ 4 倍・引き返し 0.03 倍
        w = torch.where(w.sum(-1, keepdim=True) > 0, w, torch.ones_like(w))
        return torch.multinomial(w.reshape(-1, 4).cpu(), 1, generator=self.g).view(r.shape).to(DEVICE)

    def step(self):
        self.p = self.p + NPC_SPEED
        arrive = self.p >= 1
        self.r = torch.where(arrive, self.r + DIRS4[self.d, 0], self.r)
        self.c = torch.where(arrive, self.c + DIRS4[self.d, 1], self.c)
        self.p = torch.where(arrive, self.p - 1, self.p)
        self.d = torch.where(arrive, self.choose(self.r, self.c, self.d), self.d)

    def pose(self):
        x = self.r + 0.5 + self.p * DIRS4[self.d, 0]
        y = self.c + 0.5 + self.p * DIRS4[self.d, 1]
        return torch.stack([x.float(), y.float(), self.d.float() * math.pi / 2], -1)


def people_feats(xs, ys, ths, pose):
    """プレイヤー (N,) から見た住民の数の並び (N,K,NPC_F)。世界モデルに渡すのはこれ (プロンプトではなくテンソル)"""
    c, s = torch.cos(ths)[:, None], torch.sin(ths)[:, None]
    rx, ry = pose[..., 0] - xs[:, None], pose[..., 1] - ys[:, None]
    fwd, right = rx * c + ry * s, -rx * s + ry * c
    dist = torch.sqrt(rx ** 2 + ry ** 2)
    rel = pose[..., 2] - ths[:, None]
    here = (dist < NPC_RANGE).float()
    f = torch.stack([fwd / NPC_RANGE, right / NPC_RANGE, torch.sin(rel), torch.cos(rel), dist / NPC_RANGE, here], -1)
    return f * here[..., None]

# 試しに、住民を 8 人置いた街を 4 か所から見る
if NPCS:
    torch.manual_seed(1); np.random.seed(1)
    mid = torch.tensor([0, 0, 1, N_MAPS_TRAIN], device=DEVICE)
    x, y, th = road_starts(mid)
    demo = People(mid, 8, seed=1)
    for _ in range(3):
        demo.step()
    show_rows({"目線 (住民あり)": list(render(x, y, th, mid, people=demo.pose()).permute(0, 2, 3, 1).cpu().numpy())},
              [f"地図 {m}" for m in mid.tolist()], size=2.6)

In [ ]:
# ── データ集め: 歩かせながら毎歩「描画 → DINOv2 の特徴」を記録する ──
# 1 エピソード = EP_LEN+1 フレームと EP_LEN 個の行動。全フレームを 1 列に並べて持つ。
def collect(map_ids, ep_per_map, L, batch=128):
    mids = torch.tensor([m for m in map_ids for _ in range(ep_per_map)], device=DEVICE)
    E = len(mids); M = E * (L + 1)
    feats = torch.empty(M, N_TOK, FEAT_DIM, dtype=torch.float16, device=DEVICE)
    thumbs = torch.empty(M, THUMB, THUMB, 3, dtype=torch.uint8, device="cpu" if THUMBS_ON_CPU else DEVICE)
    acts = torch.zeros(M, dtype=torch.long, device=DEVICE)
    poses = torch.zeros(M, 3, device=DEVICE)                 # 評価用に本当の位置と向きも残す
    npc = torch.zeros(M, NPCS, NPC_F, dtype=torch.float16, device=DEVICE) if NPCS else None   # プレイヤーから見た住民
    npc_pose = torch.zeros(M, NPCS, 3, device=DEVICE) if NPCS else None                      # 住民の本当の位置と向き
    for b in range(0, E, batch):
        mid = mids[b:b + batch]; n = len(mid)
        x, y, th = random_starts(mid); pol = Explorer(n)
        ppl = People(mid, NPCS) if NPCS else None
        rows = (torch.arange(b, b + n, device=DEVICE) * (L + 1))
        for t in range(L + 1):
            pp = ppl.pose() if ppl is not None else None
            img = render(x, y, th, mid, people=pp)
            feats[rows + t] = dino_encode(img, TOK_GRID).half()
            thumbs[(rows + t).to(thumbs.device)] = (F.interpolate(img, size=THUMB, mode="bilinear", antialias=True).clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).to(thumbs.device)
            poses[rows + t] = torch.stack([x, y, th], 1)
            if ppl is not None:
                npc[rows + t] = people_feats(x, y, th, pp).half(); npc_pose[rows + t] = pp
            if t < L:
                a = pol(); acts[rows + t] = a
                x, y, th, _ = step_env(x, y, th, a, mid)
                if ppl is not None: ppl.step()
    ep_start = list(range(0, M, L + 1))
    return dict(feats=feats, thumbs=thumbs, acts=acts, poses=poses, mids=mids, npc=npc, npc_pose=npc_pose,
                ep_start=ep_start, ep_len=[L + 1] * E)

t0 = time.time()
train = collect(range(N_MAPS_TRAIN), EP_PER_MAP, EP_LEN)
test = collect(range(N_MAPS_TRAIN, N_MAPS_TRAIN + N_MAPS_TEST), 4, EP_LEN)
print(f"学習 {len(train['feats']):,} フレーム / 評価 {len(test['feats']):,} フレーム ({time.time()-t0:.0f}s)")

# 特徴を学習データの平均と標準偏差で標準化する (以後はすべてこの空間で考える)
# 16×16 の特徴は 10 GB を超える。まとめて float にすると 2 倍の 20 GB がもう 1 つ要って GPU に載らないので、
# 少しずつ区切って計算し、その場で書き換える (余分に使うのは 1 区切り分の 1〜2 GB だけ)
@torch.no_grad()
def feat_stats(feats, chunk=1024):
    dev = "cpu" if feats.device.type == "mps" else feats.device     # Mac の GPU (MPS) は float64 を使えないので CPU で足す
    s = torch.zeros(FEAT_DIM, dtype=torch.float64, device=dev); s2 = torch.zeros_like(s); n = 0
    for c in feats.split(chunk):
        c = c.to(dev).double().flatten(0, 1); s += c.sum(0); s2 += (c * c).sum(0); n += len(c)
    mu = s / n
    return mu.float().to(feats.device), ((s2 - n * mu * mu) / (n - 1)).clamp_min(0).sqrt().float().to(feats.device)

FEAT_MU, FEAT_SD = feat_stats(train["feats"])
FEAT_SD = FEAT_SD + 1e-4
with torch.no_grad():
    for d in (train, test):
        for c in d["feats"].split(1024):      # split は元の一部を指すので、copy_ で元のテンソルが書き換わる
            c.copy_(((c.float() - FEAT_MU) / FEAT_SD).half())
if DEVICE == "cuda":
    print(f"GPU のメモリ: 使用中 {torch.cuda.memory_allocated() / 1e9:.1f} GB / 全体 {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def observe(x, y, th, mid):
    """本物の環境で目線を描いて、標準化した特徴にする"""
    return (dino_encode(render(x, y, th, mid), TOK_GRID) - FEAT_MU) / FEAT_SD

## 学習する

In [ ]:
wm = WorldModel(FEAT_DIM, N_TOK, HIST, n_actions=len(ACTIONS), d=WM_D, depth=WM_DEPTH, heads=WM_HEADS,
                n_npc=NPCS, npc_f=NPC_F).to(DEVICE)
print(f"世界モデル {sum(p.numel() for p in wm.parameters())/1e6:.1f}M パラメータ / {WM_STEPS} 回 / ROLLOUT {ROLLOUT}")
starts = valid_starts(train["ep_start"], train["ep_len"], HIST + ROLLOUT - 1)
losses = train_wm(wm, train["feats"], train["acts"], starts, WM_STEPS, batch=WM_BATCH, lr=WM_LR, rollout=ROLLOUT,
                  npc=train["npc"])
plt.figure(figsize=(6, 3)); plt.plot(np.convolve(losses, np.ones(50) / 50, "valid")); plt.yscale("log")
plt.title("世界モデルの学習 (1 歩先の特徴の二乗誤差)"); plt.xlabel("step"); plt.show()

In [ ]:
dec = Decoder(FEAT_DIM, TOK_GRID, ch=DEC_CH, out=THUMB, res=DEC_RES, ctx=DEC_CTX).to(DEVICE)
print(f"復元器 {sum(p.numel() for p in dec.parameters())/1e6:.1f}M パラメータ / {DEC_STEPS} 回 / "
      f"見た目 {DEC_PERCEPTUAL} / 本物らしさ {DEC_GAN} / 手がかり {DEC_CTX} / ノイズ {DEC_NOISE} / 想像 {DEC_IMAGINED}")
train_decoder(dec, train, DEC_STEPS, batch=DEC_BATCH, perceptual=DEC_PERCEPTUAL, gan=DEC_GAN, noise=DEC_NOISE,
              imagined=DEC_IMAGINED, ema=DEC_EMA, wm=wm)
i = torch.arange(0, len(test["feats"]), max(1, len(test["feats"]) // 8))[:8]
prev = torch.maximum(frame_starts(test)[i], i - 1)          # 手がかり: 1 歩前の本物の画像
show_rows({"実際": list(thumbs_at(test, i).cpu().numpy() / 255),
           "特徴から復元": list(decode(dec, test["feats"][i.to(DEVICE)], ctx=thumbs_at(test, prev)))},
          size=1.8, title="見たことのない街で、特徴 → 画像 の復元 (これが「想像」を見るときの画質の上限)")

## 1. どのくらい先まで当たるか

評価用の街 (学習で一度も見ていない) で、実際に見た直近 3 フレームと、その先の行動だけを渡して 15 歩先まで想像させ、
本物の特徴と比べます。比べる相手は「何も変わらないと予想する」(最後に見たフレームをそのまま使う) 素朴なやり方です。

In [ ]:
K = 15
def eval_windows(data, n_per_ep=4):
    """各エピソードから (文脈の先頭 i) を選ぶ。i..i+HIST-1 が文脈、その先 K 歩が答え"""
    out = []
    for s, n in zip(data["ep_start"], data["ep_len"]):
        hi = n - HIST - K
        if hi > 0:
            out += [s + int(o) for o in np.linspace(0, hi - 1, n_per_ep)]
    return torch.tensor(out, device=DEVICE)

w = eval_windows(test)
ctx = test["feats"][w[:, None] + torch.arange(HIST, device=DEVICE)].float()
acts_seq = test["acts"][w[:, None] + torch.arange(HIST - 1 + K, device=DEVICE)]
truth = test["feats"][w[:, None] + HIST + torch.arange(K, device=DEVICE)].float()
n_eval = test["npc"][w[:, None] + 1 + torch.arange(HIST - 1 + K, device=DEVICE)].float() if NPCS else None
pred = imagine(wm, ctx, acts_seq, n_eval)
err_wm = ((pred - truth) ** 2).mean((0, 2, 3)).cpu().numpy()
err_copy = ((ctx[:, -1:] - truth) ** 2).mean((0, 2, 3)).cpu().numpy()

plt.figure(figsize=(6, 3.2))
plt.plot(range(1, K + 1), err_copy, "o-", label="何も変わらないと予想")
plt.plot(range(1, K + 1), err_wm, "o-", label="世界モデルの想像")
plt.xlabel("何歩先か"); plt.ylabel("特徴の二乗誤差 (小さいほど良い)"); plt.legend(); plt.grid(alpha=.3)
plt.title("見たことのない街での予測"); plt.show()
print("1 歩先:  世界モデル %.3f / 何も変わらない %.3f" % (err_wm[0], err_copy[0]))
print("%d 歩先: 世界モデル %.3f / 何も変わらない %.3f" % (K, err_wm[-1], err_copy[-1]))

# ぼやけの原因を切り分ける: 「本物の特徴を復元」がぼやけるなら復元器の限界、「想像」だけがぼやけるなら世界モデルのずれ
# 手がかりの画像は、どちらも最後に本物を見たコマ (想像を始めた所) を使う
last_real = thumbs_at(test, (w + HIST - 1).cpu())                       # (W,H,W,3)
def pix_err(z_all):
    """各歩の、復元した画像と本物の画像の画素の差 (L1)"""
    errs = []
    for k in range(K):
        y = decode(dec, z_all[:, k], ctx=last_real)
        x = thumbs_at(test, (w + HIST + k).cpu()).cpu().numpy() / 255
        errs.append(np.abs(y - x).mean())
    return np.array(errs)
pe_rec, pe_wm = pix_err(truth), pix_err(pred)
plt.figure(figsize=(6, 3.2))
plt.plot(range(1, K + 1), pe_rec, "o-", label="本物の特徴を復元 (復元器の限界)")
plt.plot(range(1, K + 1), pe_wm, "o-", label="想像した特徴を復元")
plt.xlabel("何歩先か"); plt.ylabel("画素の差 (小さいほど良い)"); plt.legend(); plt.grid(alpha=.3)
plt.title("ぼやけはどこから来るか"); plt.show()

# 想像を画像にして見比べる
j = 0
steps = [0, 1, 3, 7, 11, 14]
real = thumbs_at(test, (w[j] + HIST + torch.tensor(steps, device=DEVICE)).cpu()).cpu().numpy() / 255
show_rows({"実際": list(real), "本物の特徴を復元": list(decode(dec, truth[j, steps], ctx=last_real[j])),
           "想像": list(decode(dec, pred[j, steps], ctx=last_real[j]))},
          [f"{s+1} 歩先" for s in steps], size=1.8,
          title="行動: " + " ".join(ACTIONS[a] for a in acts_seq[j, HIST - 1:].tolist()))
display(animate_rows({"実際": thumbs_at(test, (w[j] + HIST + torch.arange(K, device=DEVICE)).cpu()).cpu().numpy() / 255,
                      "想像": decode(dec, pred[j], ctx=last_real[j])}))

## 2. 「もし別の行動をしたら」を想像する

同じ場所から、行動の計画だけを変えて想像させます。本物の描画器でも同じ行動を実行して並べます。
建物に向かってまっすぐ進む計画では、ぶつかって景色が止まるところまで想像できているかが見どころです。

In [ ]:
PLANS = {"ずっと前進": [0] * 12, "ずっと左": [1] * 12, "ずっと右": [2] * 12, "左 6 → 右 6": [1] * 6 + [2] * 6,
         "前進 6 → 後退 6": [0] * 6 + [3] * 6}

def true_rollout(x, y, th, mid, plan):
    """本物の環境で実行して、各歩の特徴と縮小画像を返す"""
    fs, ims = [], []
    for a in plan:
        x, y, th, _ = step_env(x, y, th, torch.tensor([a], device=DEVICE), mid)
        img = render(x, y, th, mid)
        fs.append((dino_encode(img, TOK_GRID) - FEAT_MU) / FEAT_SD)
        ims.append(F.interpolate(img, size=THUMB, mode="bilinear", antialias=True)[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy())
    return torch.cat(fs), np.stack(ims)

# 評価用の街で、文脈の 3 フレームはまっすぐ前進して集める
torch.manual_seed(3); np.random.seed(3)
mid = torch.tensor([N_MAPS_TRAIN], device=DEVICE)
x, y, th = random_starts(mid)
ctx = [observe(x, y, th, mid)]
for _ in range(HIST - 1):
    x, y, th, _ = step_env(x, y, th, torch.tensor([0], device=DEVICE), mid)
    ctx.append(observe(x, y, th, mid))
ctx = torch.stack(ctx, 1)
ctx_img = F.interpolate(render(x, y, th, mid), size=THUMB, mode="bilinear", antialias=True)[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy()
steps = [0, 2, 5, 8, 11]
anim_rows = {}
for name, plan in PLANS.items():
    a = torch.tensor([[0] * (HIST - 1) + plan], device=DEVICE)
    imag = imagine(wm, ctx, a)[0]
    _, real = true_rollout(x, y, th, mid, plan)
    show_rows({"実際": list(real[steps]), "想像": list(decode(dec, imag[steps], ctx=ctx_img))},
              [f"{s+1} 歩先" for s in steps], size=1.7, title=name)
    anim_rows[name + " 想像"] = decode(dec, imag, ctx=ctx_img)
display(animate_rows(anim_rows))

## 2.5 ほかの住民 (NPC) と一緒に住めるか

設定のセルで `NPCS` を 1 以上にしたときだけ動きます (0 なら飛ばします)。

住民の位置は**シミュレーターが決める本当の状態**で、世界モデルには「プレイヤーから見た住民の数の並び」(1 人 6 個の数: 前への距離・右への距離・向きの sin と cos・距離・いるか) を**テンソルで**渡します。
渡すのは、各行動の「あと」に住民がどこにいるかです。住民がどう動くかを世界モデルが当てる必要はなく、「そこにいるときの見た目」を描くだけで済みます。

1. **数で比べる**: 住民が写っている画素だけで、想像と本物の差を比べます。配列を渡したときに差が小さければ、配列が効いています
2. **目で見る**: 住民が見える場所で、少し進んでから右を向き、また前に戻ります。戻ったときに、同じ色の人が同じ所にいれば、住民と一緒に住めています (配列を渡さないと、住民は消えるか、ぼやけた影になります)

この節の街では、プレイヤーは住民とぶつかりません (通り抜けます)。2 と 4 の節は、住民のいない街で試しています。

In [ ]:
# NPCS (ほかの住民の数) が 0 なら飛ばす
if not NPCS:
    print("NPCS = 0 なので飛ばします (設定のセルで NPCS = 5 などにすると、住民と一緒に学習します)")
else:
    def frame_mid(data, idx):
        """フレームの番号 → 地図の番号 (どのエピソードも同じ長さ EP_LEN+1)"""
        return data["mids"][idx // (EP_LEN + 1)]

    # ── (1) 数で比べる: 住民が写っている画素だけで、想像と本物の差を比べる (住民の配列を渡す / 渡さない) ──
    Wn = w[:48]
    ctx_n = test["feats"][Wn[:, None] + torch.arange(HIST, device=DEVICE)].float()
    a_n = test["acts"][Wn[:, None] + torch.arange(HIST - 1 + K, device=DEVICE)]
    n_seq = test["npc"][Wn[:, None] + 1 + torch.arange(HIST - 1 + K, device=DEVICE)].float()
    pred_with = imagine(wm, ctx_n, a_n, n_seq)
    pred_wo = imagine(wm, ctx_n, a_n, torch.zeros_like(n_seq))
    last_n = thumbs_at(test, (Wn + HIST - 1).cpu())
    STEPS_N = [0, 4, 9, 14]
    err_with, err_wo, frac = [], [], []
    for k in STEPS_N:
        idx = Wn + HIST + k
        P, PP, M_ = test["poses"][idx], test["npc_pose"][idx], frame_mid(test, idx)
        shrink = lambda im: F.interpolate(im, size=THUMB, mode="bilinear", antialias=True)
        diff = (shrink(render(P[:, 0], P[:, 1], P[:, 2], M_, people=PP)) - shrink(render(P[:, 0], P[:, 1], P[:, 2], M_))).abs().sum(1)
        mask = (diff > 0.15).cpu().numpy()                                       # 住民が写っている画素
        frac.append(mask.mean())
        real_k = thumbs_at(test, idx.cpu()).cpu().numpy() / 255
        for pred_, errs in ((pred_with, err_with), (pred_wo, err_wo)):
            e = np.abs(decode(dec, pred_[:, k], ctx=last_n) - real_k).mean(-1)
            errs.append(e[mask].mean() if mask.any() else np.nan)
    print("住民が写っている画素の割合 (" + " / ".join(f"{k+1} 歩先 {f_:.2%}" for k, f_ in zip(STEPS_N, frac)) + ")")
    x_ = np.arange(len(STEPS_N))
    plt.figure(figsize=(6, 3.2))
    plt.bar(x_ - 0.2, err_wo, 0.4, label="住民の配列を渡さない")
    plt.bar(x_ + 0.2, err_with, 0.4, label="住民の配列を渡す")
    plt.xticks(x_, [f"{k+1} 歩先" for k in STEPS_N]); plt.ylabel("住民の画素の差 (小さいほど良い)")
    plt.legend(); plt.title("住民の位置を、配列で世界モデルに渡す効果"); plt.show()

    # ── (2) 目で見る: 住民が 2 人以上見える場所に立ち、少し進んでから右を向き、また前に戻る ──
    tan_half = math.tan(FOV / 2)
    for seed in range(200):
        torch.manual_seed(seed); np.random.seed(seed)
        mid = torch.tensor([N_MAPS_TRAIN + seed % N_MAPS_TEST], device=DEVICE)
        x, y, th = road_starts(mid)
        ppl = People(mid, NPCS, seed=seed)
        f = people_feats(x, y, th, ppl.pose())[0] * NPC_RANGE
        seen = (f[:, 0] > 0.3) & (f[:, 1].abs() < f[:, 0] * tan_half) & (f[:, 4] < 5)
        if seen.sum() >= 2:
            break
    PLAN = [0] * 4 + [2] * 8 + [1] * 8          # 前進 4 → 右 8 (約 96 度) → 左 8 (前に戻る)
    acts_all = [0] * (HIST - 1) + PLAN
    zs, imgs, ns = [], [], []
    for t, a in enumerate([None] + acts_all):
        if a is not None:
            x, y, th, _ = step_env(x, y, th, torch.tensor([a], device=DEVICE), mid)
            ppl.step()
        pp = ppl.pose()
        img = render(x, y, th, mid, people=pp)
        zs.append((dino_encode(img, TOK_GRID) - FEAT_MU) / FEAT_SD)
        imgs.append(F.interpolate(img, size=THUMB, mode="bilinear", antialias=True)[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy())
        ns.append(people_feats(x, y, th, pp))
    ctx_v = torch.stack(zs[:HIST], 1)
    a_v = torch.tensor([acts_all], device=DEVICE)
    n_v = torch.stack(ns[1:], 1)                     # 各行動の「あと」の住民
    im_with = imagine(wm, ctx_v, a_v, n_v)[0]
    im_wo = imagine(wm, ctx_v, a_v, torch.zeros_like(n_v))[0]
    real_v = np.stack(imgs[HIST:])
    ctx_img = imgs[HIST - 1]
    steps = [0, 3, 7, 11, 15, 19]
    show_rows({"実際": list(real_v[steps]),
               "想像 (住民の配列を渡す)": list(decode(dec, im_with[steps], ctx=ctx_img)),
               "想像 (渡さない)": list(decode(dec, im_wo[steps], ctx=ctx_img))},
              [f"{s+1} 歩先" for s in steps], size=1.8,
              title="前進 4 → 右を向く 8 → 前に戻る 8。戻ったとき、同じ人が同じ所にいるか")
    display(animate_rows({"実際": real_v, "想像 (配列あり)": decode(dec, im_with, ctx=ctx_img),
                          "想像 (配列なし)": decode(dec, im_wo, ctx=ctx_img)}))

## 3. 映像から行動を当てる (逆動力学)

連続する 2 フレームの特徴だけから「その間に取った行動」を当てます。前回の説明の「映像 → 座標」にあたる部分です。
壁にぶつかって動けなかった前進は、2 枚の絵が同じになるので原理的に当てられません。

In [ ]:
inv = InverseDynamics(FEAT_DIM, N_TOK, len(ACTIONS)).to(DEVICE)
opt = torch.optim.AdamW(inv.parameters(), lr=1e-3, weight_decay=0.01)
pairs = valid_starts(train["ep_start"], train["ep_len"], 1).to(DEVICE)
for step in range(INV_STEPS):
    i = pairs[torch.randint(0, len(pairs), (256,), device=DEVICE)]
    loss = F.cross_entropy(inv(train["feats"][i].float(), train["feats"][i + 1].float()), train["acts"][i])
    opt.zero_grad(); loss.backward(); opt.step()
inv.eval()

tp = valid_starts(test["ep_start"], test["ep_len"], 1).to(DEVICE)
with torch.no_grad():
    guess = torch.cat([inv(test["feats"][c].float(), test["feats"][c + 1].float()).argmax(1) for c in tp.split(2048)])
truth_a = test["acts"][tp]
moved = (test["poses"][tp + 1, :2] - test["poses"][tp, :2]).norm(dim=1) > 1e-4
acc = (guess == truth_a).float().mean().item()
acc_moved = (guess == truth_a)[moved].float().mean().item()
major = torch.bincount(truth_a, minlength=NA).max().item() / len(truth_a)
print(f"正解率 {acc:.1%} (動けた歩だけなら {acc_moved:.1%}) / いつも一番多い行動と答えた場合 {major:.1%}")
cm = torch.zeros(NA, NA, dtype=torch.long)
for t, g in zip(truth_a.tolist(), guess.tolist()): cm[t, g] += 1
print("混同行列 (行 = 本当の行動, 列 = 当てた行動):", ACTIONS)
print(cm.numpy())

## 4. 想像の中で試して、ゴールへ向かう (計画)

ゴールの**画像だけ**を渡して、そこへ行かせます。地図も座標も渡しません。

1. ゴールは「スタートから適当に 10 歩歩いた場所の目線」です (なので必ずたどり着ける)
2. 毎歩、行動の候補 (10 歩分の並び) をたくさん想像の中で試し、ゴールの特徴に一番近づく並びを選ぶ (CEM)
3. その最初の 1 歩だけを**本物の環境で**実行して、新しく見えた絵からもう一度考え直す (MPC)

ゴールに 0.5 マス以内・向き 25° 以内まで近づけたら成功です。同じ歩数でランダムに歩いた場合と比べます。
毎歩、何千通りもの未来を想像するので、このノートブックでいちばん時間のかかるセルです。

In [ ]:
# N_TRIALS (試す回数) と POP (1 回に試す行動の並びの数) は設定のセルで決める。ここがいちばん時間のかかるセル
PLAN_T, ITERS = (4, 2) if QUICK else (10, 4)   # 何歩先まで計画するか / 候補を絞り直す回数
ELITE = max(2, POP // 8)                        # 次の候補の元にする上位の数
GOAL_STEPS, MAX_STEPS = (3, 4) if QUICK else (10, 25)

@torch.no_grad()
def cem_plan(z_hist, a_hist, z_goal):
    """z_hist: (C,N,D) 直近の特徴 / a_hist: (C-1,) その間の行動 / z_goal: (N,D) → 一番よい行動の並び"""
    probs = torch.full((PLAN_T, NA), 1 / NA, device=DEVICE)
    for _ in range(ITERS):
        seq = torch.multinomial(probs, POP, replacement=True).T                  # (POP, PLAN_T)
        a = torch.cat([a_hist[None].expand(POP, -1), seq], 1)
        zf = imagine(wm, z_hist[None].expand(POP, -1, -1, -1), a)               # (POP, PLAN_T, N, D)
        cost = ((zf - z_goal) ** 2).mean((2, 3)).min(1).values                  # 途中でゴールに着けばよい
        elite = seq[cost.topk(ELITE, largest=False).indices]
        probs = 0.9 * F.one_hot(elite, NA).float().mean(0) + 0.1 / NA              # 少しだけ探索を残す
    return elite[0]

def angle_diff(a, b):
    return abs(math.atan2(math.sin(a - b), math.cos(a - b)))

def reached(p, g):
    return math.hypot(p[0] - g[0], p[1] - g[1]) < 0.5 and angle_diff(p[2], g[2]) < math.radians(25)

def trial(seed, use_model=True):
    torch.manual_seed(seed); np.random.seed(seed)
    mid = torch.tensor([N_MAPS_TRAIN + seed % N_MAPS_TEST], device=DEVICE)
    x, y, th = random_starts(mid)
    # ゴール: スタートから 10 歩歩いた先 (十分に離れた場所になるまで引き直す)
    for _ in range(20):
        gx, gy, gth = x, y, th; pol = Explorer(1)
        for _ in range(GOAL_STEPS):
            gx, gy, gth, _ = step_env(gx, gy, gth, pol(), mid)
        g = (gx.item(), gy.item(), gth.item())
        if not reached((x.item(), y.item(), th.item()), g): break
    z_goal = observe(gx, gy, gth, mid)[0]
    zs, acts, path = [observe(x, y, th, mid)[0]], [], [(x.item(), y.item())]
    for step in range(MAX_STEPS):
        if reached((x.item(), y.item(), th.item()), g):
            return True, step, path, g, mid.item()
        if use_model:
            C = min(len(zs), HIST)                      # 直近 C フレームと、その間の C-1 個の行動
            a_hist = torch.tensor(acts[len(acts) - (C - 1):], dtype=torch.long, device=DEVICE)
            a = cem_plan(torch.stack(zs[-C:]), a_hist, z_goal)[0]
        else:
            a = torch.randint(0, NA, (), device=DEVICE)
        x, y, th, _ = step_env(x, y, th, a.view(1), mid)
        zs.append(observe(x, y, th, mid)[0]); acts.append(a.item()); path.append((x.item(), y.item()))
    return reached((x.item(), y.item(), th.item()), g), MAX_STEPS, path, g, mid.item()

t0 = time.time()
res_wm, res_rand = [], []
for s in range(N_TRIALS):
    res_wm.append(trial(s, True)); res_rand.append(trial(s, False))
    if (s + 1) % 10 == 0:
        print(f"  {s+1}/{N_TRIALS} 回  計画 {np.mean([r[0] for r in res_wm]):.0%} / ランダム {np.mean([r[0] for r in res_rand]):.0%}  ({time.time()-t0:.0f}s)")
ok_wm = np.mean([r[0] for r in res_wm]); ok_rand = np.mean([r[0] for r in res_rand])
print(f"成功率: 世界モデルで計画 {ok_wm:.0%} / ランダムに歩く {ok_rand:.0%}  ({N_TRIALS} 回ずつ, {time.time()-t0:.0f}s)")
steps_ok = [r[1] for r in res_wm if r[0]]
if steps_ok: print(f"成功したときの歩数: 平均 {np.mean(steps_ok):.1f} 歩 (ゴールはスタートから {GOAL_STEPS} 歩の場所)")

# 道のりを地図に描く (青 = 計画, 灰 = ランダム, ★ = ゴール)
fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
for a, k in zip(ax, range(min(4, N_TRIALS))):
    ok, n, path, g, m = res_wm[k]; _, _, rpath, _, _ = res_rand[k]
    p = np.array(path); rp = np.array(rpath)
    lo = np.floor(np.minimum(p.min(0), g[:2]) - 2).clip(0); hi = np.ceil(np.maximum(p.max(0), g[:2]) + 2).clip(max=GRID)
    a.imshow(MAPS[m].T, cmap=cmap, vmin=0, vmax=3, origin="lower", extent=(0, GRID, 0, GRID))
    a.plot(rp[:, 0], rp[:, 1], "-", color="#888", lw=1.5)
    a.plot(p[:, 0], p[:, 1], "-", color="#2a7fff", lw=2); a.plot(*p[0], "o", color="#2a7fff")
    a.plot(g[0], g[1], "*", color="gold", ms=14, mec="k")
    a.set_xlim(lo[0], hi[0]); a.set_ylim(lo[1], hi[1]); a.set_xticks([]); a.set_yticks([])
    a.set_title(("成功" if ok else "失敗") + f" {n} 歩", fontsize=10)
plt.show()

## 5. 学習したモデルを保存して Hugging Face に置く

ここまでの `wm` (世界モデル)・`dec` (復元器)・`inv` (逆動力学) は、**Colab のメモリの中にしかありません**。ランタイムを切ると消えます。
このセルで Google Drive の `WORK_DIR/export_<PRESET>/` に書き出し (ランタイムを切っても残ります)、`HF_MODEL_REPO` を書けば Hugging Face のモデルリポジトリにも上げます。

| ファイル | 中身 |
| --- | --- |
| `world_model.safetensors` | 世界モデル (順動力学) の重み |
| `decoder.safetensors` | 特徴 → 画像 (`THUMB`×`THUMB`) の復元器 |
| `inverse_dynamics.safetensors` | 逆動力学 |
| `feat_stats.safetensors` | DINOv2 の特徴を標準化する平均と標準偏差 (これが無いと世界モデルに入れられない) |
| `config.json` | 大きさ・行動・描画の設定。シミュレーター (Space) はここを読んで同じ条件で描く |

上げるには Hugging Face の**書き込み用トークン**が要ります。Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` として入れておくと、
`login()` がそれを使います。

In [ ]:
EXPORT_DIR = os.path.join(WORK_DIR, f"export_{PRESET}" + ("_quick" if QUICK else ""))   # Drive の上。PRESET ごとに分ける
HF_MODEL_REPO = ""    # 例: "あなたのユーザー名/mesa-world-model"。空のままなら書き出すだけ

from safetensors.torch import save_file
os.makedirs(EXPORT_DIR, exist_ok=True)
def save_sd(model, name):
    save_file({k: v.detach().float().cpu().contiguous() for k, v in model.state_dict().items()}, f"{EXPORT_DIR}/{name}")
save_sd(wm, "world_model.safetensors")
save_sd(dec, "decoder.safetensors")
save_sd(inv, "inverse_dynamics.safetensors")
save_file({"mu": FEAT_MU.float().cpu().contiguous(), "sd": FEAT_SD.float().cpu().contiguous()}, f"{EXPORT_DIR}/feat_stats.safetensors")

config = dict(
    format_version=1,
    actions=ACTIONS,                       # 行動の番号 = この並び
    feat_dim=FEAT_DIM, tok_grid=TOK_GRID, hist=HIST,
    world_model=dict(d=WM_D, depth=WM_DEPTH, heads=WM_HEADS, **({"n_npc": NPCS, "npc_f": NPC_F} if NPCS else {})),
    **({"npc": dict(count=NPCS, speed=NPC_SPEED, range=NPC_RANGE, person_h=PERSON_H, person_w=PERSON_W,
                    colors=PERSON_COLORS)} if NPCS else {}),
    # res と ctx は、使うときだけ書く (前の Space の models.py でも、fast で作った重みはそのまま読める)
    decoder=dict(ch=DEC_CH, out=THUMB, **({"res": DEC_RES} if DEC_RES else {}), **({"ctx": True} if DEC_CTX else {})),
    inverse_dynamics=dict(r=32),
    img=IMG, thumb=THUMB, grid=GRID, move=MOVE, rot_deg=math.degrees(ROT),
    render=dict(fov_deg=math.degrees(FOV), eye=EYE, ssaa=SSAA, layers=LAYERS, tlayers=TLAYERS,
                tw=TW, th=TH, gt=GT, at=AT, tree_h=TREE_H, tree_r=TREE_R, fog=[FOG0, FOG1],
                textures=RAW, p_big=P_BIG),
    maps=dict(train_seeds=MAP_SEEDS[:N_MAPS_TRAIN], test_seeds=MAP_SEEDS[N_MAPS_TRAIN:]),
    dino="facebookresearch/dinov2:dinov2_vits14",
    train=dict(preset=PRESET, quick=QUICK, **{k: v for k, v in CFG.items() if k not in ("N_MAPS_TRAIN", "N_MAPS_TEST")}),
)
with open(f"{EXPORT_DIR}/config.json", "w") as f:
    json.dump(config, f, ensure_ascii=False, indent=1)
print("書き出し:", EXPORT_DIR, sorted(os.listdir(EXPORT_DIR)))

if HF_MODEL_REPO:
    from huggingface_hub import HfApi, login
    login()                                # シークレットの HF_TOKEN か、入力欄のトークンを使う
    api = HfApi()
    api.create_repo(HF_MODEL_REPO, exist_ok=True)
    api.upload_folder(folder_path=EXPORT_DIR, repo_id=HF_MODEL_REPO, commit_message="world model weights")
    print(f"アップロードしました: https://huggingface.co/{HF_MODEL_REPO}")
else:
    print("HF_MODEL_REPO が空なので、アップロードはしていません (書き出したものは Google Drive に残っています)")

## うまくいかないときに触るところ

どれも先頭の**設定のセル**で変えます (`PRESET` を上げるか、`OVERRIDE` に書く)。

- **復元器の絵 (実際 / 特徴から復元) の時点でぼやけている**: 世界モデルではなく復元の上限の問題。
  `TOK_GRID = 16` と `THUMB = 128` にする、`DEC_STEPS` を増やす、`DEC_PERCEPTUAL` を上げる (0.2 → 0.5。上げすぎると色がずれる)
- **1 歩先はくっきりなのに、先に行くほどぼやける**: 世界モデルのずれが積もっている。`WM_STEPS` を増やす、
  `ROLLOUT` を 2 → 3 にする、`EP_PER_MAP` を増やしてデータを足す、`WM_D` / `WM_DEPTH` を大きくする
- **GPU のメモリが足りない (out of memory)**: `WM_BATCH` を半分にする、`EP_PER_MAP` か `ROLLOUT` を減らす
- **計画の成功率が低い**: `POP` を増やす。ゴールの特徴に近づくのに位置が合わないときは、
  見た目が似た場所を取り違えています (特徴だけで考える方式の限界。過去のフレームも手がかりに使うと改善します)

## 次の一歩

- 学習した `wm` を住民の頭に入れる: 「目的の店の外観 (画像) をゴールにして、地図なしで歩く」が、そのまま試せる形になっています
- 行動にスキル (walk / carry / stroll) や速さを足すと、BC の行動空間と揃います
- 書き出したモデルを Hugging Face の Space (Gradio) に載せると、前進・左・右・後退のボタンで「想像の中を歩く」シミュレーターになります (`hf/` フォルダ)